pai_nosso_refatorado_v1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/alanabdmorais/pai_nosso_refatorado_v1/blob/main/pai_nosso_refatorado_v1.ipynb

# 🎬 Pipeline Multilíngue v1 — Orações com Legendas Morfológicas

**Estratégia de classificação intermediária:**
O Groq gera as classificações → você revisa com uma IA → pipeline continua com os JSONs corrigidos.

---

### Fluxo completo

| # | Fase | O que faz | Ação |
|---|------|-----------|------|
| 0 | Setup | Instala deps, monta Drive, importa módulos | Automático |
| Init | — | Cria config, groq e pipeline | Automático |
| B0 | YouTube | Baixa legendas de referência (opcional) | Manual |
| 1 | Áudio | Edge TTS → .wav no Drive | Automático |
| 2 | Whisper | Transcrição → SRT mestre (timestamps + segmentação) | Automático |
| 3 | Correção PT | Groq corrige o SRT | Automático |
| 4 | Traduções | Groq gera EN/ES/FR | Automático |
| **5A** | **Classificação** | **Groq classifica + gera pacote de revisão → pausa** | **Automático → pausa** |
| **5B** | **Revisão** | **Você corrige JSONs com IA externa** | **👤 Manual** |
| **5C** | **Recarregar** | **Pipeline carrega JSONs corrigidos** | **Automático** |
| 6 | Clipes | Baixa e corta vídeos do Pixabay | Automático |
| 7 | Vídeo base | Crédito + narração + trilha | Automático |
| 8 | Legendas ASS | Queima legendas coloridas no vídeo | Automático |

> **Retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`

In [4]:
#conectar ao Git Hub
!git config --global user.email "alanabdmorais@gmail.com"
!git config --global user.name "Alana"

In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 LIMPAR PONTO DE MONTAGEM ANTES DO SETUP                     ║
# ╚══════════════════════════════════════════════════════════════════╝

import os
import shutil

# Forçar desmontagem
!fusermount -u /content/drive 2>/dev/null

# Limpar o diretório se tiver arquivos
if os.path.exists('/content/drive'):
    try:
        shutil.rmtree('/content/drive')
        print('✅ Diretório /content/drive removido')
    except:
        print('⚠️ Não foi possível remover, mas vamos tentar montar mesmo assim')

print('\n✅ Pronto para montar o Drive novamente')


✅ Pronto para montar o Drive novamente


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup (rode uma vez por sessão)                    ║
# ╚══════════════════════════════════════════════════════════════════╝

# Sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp nest_asyncio
print('✅ pacotes Python')

# Drive - FORÇAR REMONTAGEM
from google.colab import drive, userdata

# Tentar desmontar se já estiver montado
try:
    drive.flush_and_unmount()
    print('🔄 Desmontando Drive anterior...')
except:
    pass

# Montar com force_remount
drive.mount('/content/drive', force_remount=True)
print('✅ Drive montado')

# Copiar módulos do Drive para /content/pipeline
import shutil, os, sys, logging
from pathlib import Path

# CAMINHO PADRONIZADO: usando pasta modulos
PASTA_MODULOS = '/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/modulos'

# Fallback para o caminho antigo (caso não encontre)
if not os.path.exists(PASTA_MODULOS):
    PASTA_MODULOS_FALLBACK = '/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/modulos'
    if os.path.exists(PASTA_MODULOS_FALLBACK):
        PASTA_MODULOS = PASTA_MODULOS_FALLBACK
        print(f'⚠️ Usando fallback: {PASTA_MODULOS}')

DESTINO = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    # Remove destino antigo se existir
    if os.path.exists(DESTINO):
        shutil.rmtree(DESTINO)

    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')

    # Listar arquivos copiados
    print('\n📁 Arquivos copiados:')
    for py_file in sorted(Path(DESTINO).glob('*.py')):
        print(f'   📄 {py_file.name}')
else:
    print(f'❌ Pasta não encontrada: {PASTA_MODULOS}')
    print('   Verifique se os arquivos .py estão em:')
    print('   /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/modulos/')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('\n✅ Setup concluído!')

✅ ffmpeg + espeak-ng
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 4.6 MB/s eta 0:00:00
✅ pacotes Python
Drive not mounted, so nothing to flush and unmount.
🔄 Desmontando Drive anterior...
Mounted at /content/drive
✅ Drive montado
✅ Módulos copiados: /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/modulos → /content/pipeline

📁 Arquivos copiados:
   📄 __init__.py
   📄 checkpoint.py
   📄 classification.py
   📄 config.py
   📄 constants.py
   📄 drive_utils.py
   📄 ffmpeg_utils.py
   📄 groq_client.py
   📄 models.py
   📄 srt_utils.py
   📄 video_pipeline.py

✅ Setup concluído!


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAÇÃO DO PIPELINE (com correção de path)            ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path

# 🔧 CORREÇÃO: Adicionar o caminho dos módulos
caminho_modulos = '/content/pipeline'
if caminho_modulos not in sys.path:
    sys.path.insert(0, caminho_modulos)
    print(f"✅ Caminho adicionado: {caminho_modulos}")

# Verificar se os arquivos existem
from pathlib import Path
pipeline_dir = Path('/content/pipeline')
if pipeline_dir.exists():
    py_files = list(pipeline_dir.glob('*.py'))
    print(f"📁 {len(py_files)} arquivos encontrados em /content/pipeline/")
    for py in py_files[:5]:
        print(f"   📄 {py.name}")
else:
    print(f"❌ Pasta /content/pipeline/ não encontrada!")

# Importar módulos
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# Configuração
config = PipelineConfig(
    NOME_ORACAO = 'pai_nosso',
    # Para outra oração:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',

    # Modo de vídeo:
    # False (padrão) = cores por palavra (classificação morfológica)
    # True           = cor sólida por idioma (pula classificação)
    VIDEO_SIMPLES_SEM_MORFOLOGIA = False,
)

# Cliente Groq
groq = GroqClient(
    api_key=userdata.get('GROQ_KEY'),
    nome_oracao=config.NOME_ORACAO
)

# Pipeline
pipeline = VideoPipeline(config, groq)

# Verificar checkpoint
cp = Checkpoint()
print("\n" + "="*60)
print("📋 CONFIGURAÇÃO DO PIPELINE")
print("="*60)
print(config.resumo())
print()
print(cp.resumo())
print(f"\n▶ Próxima fase: {cp.proxima_fase_pendente() or '(tudo concluído)'}")
print("="*60)

print("\n✅ Pipeline inicializado com sucesso!")

📁 11 arquivos encontrados em /content/pipeline/
   📄 config.py
   📄 checkpoint.py
   📄 drive_utils.py
   📄 __init__.py
   📄 classification.py

📋 CONFIGURAÇÃO DO PIPELINE
Oração:        pai_nosso
Áudio:         pai_nosso_audio.wav
SRT mestre:    pai_nosso_pt.srt
Vídeo final:   pai_nosso_final.mp4
Idiomas:       pt, en, es, fr
Voz Edge TTS:  pt-BR-AntonioNeural
Modelo Groq:   llama-3.3-70b-versatile
Duração clipe: 5s
Assets:        /content/drive/MyDrive/pai_nosso_refatorado_v1/assets
Correcoes:     /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/pai_nosso
Brutos:        /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/brutos/pai_nosso
Modo simples:  NÃO (completo)

── Checkpoint ──────────────────────────────
  audio_gerado                 ⬜ pendente
  srt_pt_bruto                 ⬜ pendente
  srt_pt_corrigido             ⬜ pendente
  srt_traduzidos               ⬜ pendente
  classificacoes_feitas        ⬜ pendente
  clipes_cortados              ⬜ pendente
  vid

In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA (COM CAMINHOS CORRIGIDOS)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil
import sys
import os

# CAMINHOS CORRIGIDOS (apontando para o repositório consolidado)
BASE_DRIVE = Path('/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline')
PASTA_BRUTOS = BASE_DRIVE / 'brutos'
PASTA_CORRECOES = BASE_DRIVE / 'correcoes'

def limpeza_seletiva():
    print("═" * 65)
    print("🧹 LIMPEZA SELETIVA")
    print("═" * 65)
    print("\nEscolha o que limpar (digite os números separados por vírgula)")
    print("Exemplo: 1,3,5")
    print("═" * 65)
    print("  1 - 🎵 Áudios (.wav, .mp3)")
    print("  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]")
    print("  3 - 🎬 Vídeos gerados (base, final, clipes)")
    print("  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)")
    print("  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)")
    print("  6 - 📌 Checkpoint (checkpoint.json)")
    print("  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)")
    print("  8 - 📦 Cache Python (módulos carregados)")
    print("  9 - 🗑️ Cache FASE 3 (fase3_cache.json)")
    print(" 10 - 🗑️ Cache Classificação (cache_classificacao/)")
    print(" 11 - 🔥 LIMPAR TUDO (exceto Drive)")
    print(" 12 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)")
    print("  0 - Sair")
    print("═" * 65)

    opcoes = input("\nDigite sua escolha: ").strip()

    if opcoes == '0':
        print("Saindo...")
        return

    opcoes_lista = [int(x.strip()) for x in opcoes.split(',')]
    cont = 0

    # 1. Áudios
    if 1 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n🎵 Removendo áudios...")
        for f in Path('.').glob('*_audio.wav'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.mp3'):
            if 'musica' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 2. Legendas
    if 2 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n📝 Removendo legendas...")
        for f in Path('.').glob('*.srt'):
            if 'yt_ref' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.ass'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")

    # 3. Vídeos
    if 3 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n🎬 Removendo vídeos...")
        patterns = ['*_base.mp4', '*_final.mp4', 'clipe_*.mp4', 'temp_*.mp4', 'video_*.mp4', 'raw_*.mp4']
        for pattern in patterns:
            for f in Path('.').glob(pattern):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 4. JSONs BRUTOS (pasta brutos/ no Drive)
    if 4 in opcoes_lista or 12 in opcoes_lista:
        print("\n📊 Removendo JSONs BRUTOS...")
        if PASTA_BRUTOS.exists():
            for oracao_dir in PASTA_BRUTOS.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ brutos/{oracao_dir.name}/{f.name}")
        else:
            print("   ℹ️ Pasta brutos/ não encontrada")

    # 5. JSONs CORRIGIDOS (pasta correcoes/ no Drive)
    if 5 in opcoes_lista or 12 in opcoes_lista:
        print("\n✅ Removendo JSONs CORRIGIDOS...")
        if PASTA_CORRECOES.exists():
            for oracao_dir in PASTA_CORRECOES.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ correcoes/{oracao_dir.name}/{f.name}")
        else:
            print("   ℹ️ Pasta correcoes/ não encontrada")

    # 6. Checkpoint
    if 6 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n📌 Removendo checkpoint...")
        cp = Path('checkpoint.json')
        if cp.exists():
            cp.unlink()
            cont += 1
            print("   🗑️ checkpoint.json")

    # 7. Pastas temporárias
    if 7 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n📁 Removendo pastas temporárias...")
        pastas = ['clipes_cortados', 'clipes_prontos', 'temp_raw', '__pycache__']
        for pasta in pastas:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p)
                cont += 1
                print(f"   🗑️ {pasta}/")

    # 8. Cache Python
    if 8 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        print("\n📦 Limpando cache Python...")
        modulos = ['groq_client', 'video_pipeline', 'config', 'classification',
                   'checkpoint', 'drive_utils', 'ffmpeg_utils', 'srt_utils',
                   'models', 'constants', 'pipeline']
        for m in modulos:
            if m in sys.modules:
                del sys.modules[m]
                cont += 1
                print(f"   🗑️ {m}")

    # 9. Cache FASE 3
    if 9 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        cache_fase3 = Path('/content/fase3_cache.json')
        if cache_fase3.exists():
            cache_fase3.unlink()
            cont += 1
            print("   🗑️ fase3_cache.json")

    # 10. Cache Classificação
    if 10 in opcoes_lista or 11 in opcoes_lista or 12 in opcoes_lista:
        cache_classif = Path('/content/cache_classificacao')
        if cache_classif.exists():
            shutil.rmtree(cache_classif)
            cont += 1
            print("   🗑️ cache_classificacao/")

    print("\n" + "═" * 65)
    print(f"✅ LIMPEZA CONCLUÍDA! {cont} item(ns) removido(s)")
    print("═" * 65)

# Executar
limpeza_seletiva()

═════════════════════════════════════════════════════════════════
🧹 LIMPEZA SELETIVA
═════════════════════════════════════════════════════════════════

Escolha o que limpar (digite os números separados por vírgula)
Exemplo: 1,3,5
═════════════════════════════════════════════════════════════════
  1 - 🎵 Áudios (.wav, .mp3)
  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]
  3 - 🎬 Vídeos gerados (base, final, clipes)
  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)
  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)
  6 - 📌 Checkpoint (checkpoint.json)
  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)
  8 - 📦 Cache Python (módulos carregados)
  9 - 🗑️ Cache FASE 3 (fase3_cache.json)
 10 - 🗑️ Cache Classificação (cache_classificacao/)
 11 - 🔥 LIMPAR TUDO (exceto Drive)
 12 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)
  0 - Sair
═════════════════════════════════════════════════════════════════

Digite sua escolha: 12

🎵 Removendo áudios...

📝 Removendo legendas...

🎬 Removendo vídeos...

📊 Removendo JSO

### 🔵 Célula B0 — Opcional: baixar legendas do YouTube
Execute **antes** das fases 1–8 para ter traduções mais fiéis. Se pular, o Groq traduz a partir do PT.

In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — Baixar legendas do YouTube                        ║
# ║  (BAIXA TAMBÉM A REFERÊNCIA PT para a FASE 3)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
import shutil
from drive_utils import DriveClient

cfg   = config
drive = DriveClient.get()

drive.download_se_ausente(cfg.pasta_assets, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

print("═" * 60)
print("📺 BAIXANDO LEGENDAS DO YOUTUBE")
print("═" * 60)

# ── COLOQUE AS URLs DOS VÍDEOS COM LEGENDAS ABAIXO ────────────────────────────
URLS = {
    'pt': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'en': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'es': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'fr': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
}

for lang, url in URLS.items():
    nome_srt = cfg.nome_srt(lang)
    print(f'\n⬇️  Baixando {lang.upper()} de: {url[:50]}...')
    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang, '--write-auto-sub',
        '--skip-download', '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}', *cookies_flag, url
    ]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    encontrado = False
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt} salvo')
        encontrado = True
        break
    if not encontrado:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')

# Criar referência PT para a FASE 3
print("\n" + "═" * 60)
print("📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)")
print("═" * 60)
srt_pt   = Path(cfg.nome_srt('pt'))
ref_path = Path(f'{cfg.NOME_ORACAO}_yt_ref_pt.srt')
if srt_pt.exists():
    shutil.copy(srt_pt, ref_path)
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f'✅ Referência PT salva: {ref_path}  ({len(ref_legendas)} segmentos)')
    print(f'   Preview: {ref_legendas[0].texto[:80]}')
else:
    print(f'⚠️  SRT PT não encontrado. A FASE 3 corrigirá sem referência.')
print("═" * 60)

════════════════════════════════════════════════════════════
📺 BAIXANDO LEGENDAS DO YOUTUBE
════════════════════════════════════════════════════════════

⬇️  Baixando PT de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_pt.srt salvo

⬇️  Baixando EN de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_en.srt salvo

⬇️  Baixando ES de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_es.srt salvo

⬇️  Baixando FR de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_fr.srt salvo

════════════════════════════════════════════════════════════
📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)
════════════════════════════════════════════════════════════
✅ Referência PT salva: pai_nosso_yt_ref_pt.srt  (11 segmentos)
   Preview: Pai nosso que estais no  céu,
════════════════════════════════════════════════════════════


### ▶ Fases 1–4 — Automáticas

In [9]:
# ── FASE 1: Áudio com Edge TTS ───────────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')



✅ pai_nosso_audio.wav  (162 KB)


In [10]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 2 — Whisper (MESTRE de timestamps e segmentação)          ║
# ╚══════════════════════════════════════════════════════════════════╝

print('═' * 65)
print('🎙️ FASE 2 — WHISPER (mestre de timestamps e segmentação)')
print('   O Whisper segmenta por pausas naturais do áudio.')
print('   O texto será aprimorado na Fase 3 com referência do YouTube.')
print('═' * 65)

srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
legs_w = ler_srt(srt_bruto)
print(f'\n✅ Whisper: {len(legs_w)} segmentos')
for leg in legs_w:
    print(f'  [{leg.id:02d}]  {leg.inicio_str} → {leg.fim_str}  |  {leg.texto}')


═════════════════════════════════════════════════════════════════
🎙️ FASE 2 — WHISPER (mestre de timestamps e segmentação)
   O Whisper segmenta por pausas naturais do áudio.
   O texto será aprimorado na Fase 3 com referência do YouTube.
═════════════════════════════════════════════════════════════════


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 85.0MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



✅ Whisper: 7 segmentos
  [01]  00:00:00,000 → 00:00:04,460  |  Pai nosso que estáis no céu, santificados sejam o vosso nome.
  [02]  00:00:05,540 → 00:00:06,880  |  Vem a nós o vosso reino.
  [03]  00:00:07,800 → 00:00:11,980  |  Seja feita a vossa vontade, assim na Terra como no céu.
  [04]  00:00:12,940 → 00:00:15,020  |  O pão nosso de cada dia nos dá hoje.
  [05]  00:00:15,980 → 00:00:17,720  |  Perdua aí as nossas ofensas.
  [06]  00:00:18,600 → 00:00:21,400  |  Assim como nós perduamos aqui nos tem ofendido.
  [07]  00:00:22,180 → 00:00:26,700  |  E não nos deixe cair em tentação, mas livrar-nos do mal a mim.


In [11]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🗑️ REMOVER CACHE CORROMPIDO                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

# Remover cache corrompido da FASE 3
cache_pt = Path('/content/pai_nosso_pt_corrigido.json')
if cache_pt.exists():
    tamanho = cache_pt.stat().st_size
    if tamanho == 0:
        cache_pt.unlink()
        print("✅ Cache corrompido (vazio) removido!")
    else:
        print(f"⚠️ Cache tem {tamanho} bytes - mantido")

# Remover cache de traduções se existir e estiver corrompido
cache_traducoes = Path('/content/pai_nosso_traducoes.json')
if cache_traducoes.exists():
    tamanho = cache_traducoes.stat().st_size
    if tamanho == 0:
        cache_traducoes.unlink()
        print("✅ Cache de traduções corrompido removido!")

print("\n✅ Pronto! Agora execute a FASE 3")


✅ Pronto! Agora execute a FASE 3


In [12]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 CÉLULA DE DIAGNÓSTICO - PROVA REAL                          ║
# ║  Mostra Whisper original e YouTube original para comparação     ║
# ║  GENÉRICO - funciona para qualquer oração                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt
import re

print("═" * 80)
print("🔍 DIAGNÓSTICO - TEXTOS ORIGINAIS")
print(f"   Oração: {config.NOME_ORACAO}")
print("═" * 80)

# ═══════════════════════════════════════════════════════════════════
# 1. WHISPER ORIGINAL
# ═══════════════════════════════════════════════════════════════════

srt_whisper = Path(f'{config.NOME_ORACAO}_pt_edge.srt')
if not srt_whisper.exists():
    for nome in [f'{config.NOME_ORACAO}_pt_whisper.srt', f'{config.NOME_ORACAO}_pt_edge.srt']:
        if Path(nome).exists():
            srt_whisper = Path(nome)
            break

print("\n📁 1. WHISPER (FASE 2)")
print("─" * 80)

if srt_whisper.exists():
    legendas_whisper = ler_srt(srt_whisper)
    print(f"   Segmentos: {len(legendas_whisper)}")
    print()
    for leg in legendas_whisper:
        print(f"   [{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"        {leg.texto}")
        print()
else:
    print("   ❌ Whisper não encontrado!")

# ═══════════════════════════════════════════════════════════════════
# 2. YOUTUBE ORIGINAL
# ═══════════════════════════════════════════════════════════════════

srt_ref = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')

print("\n📁 2. YOUTUBE (REFERÊNCIA - célula B0)")
print("─" * 80)

if srt_ref.exists():
    legendas_ref = ler_srt(srt_ref)
    print(f"   Segmentos: {len(legendas_ref)}")
    print()
    for leg in legendas_ref:
        print(f"   [{leg.id:02d}] {leg.texto}")
    print()

    # Texto completo do YouTube
    texto_youtube_completo = " ".join([leg.texto for leg in legendas_ref])
    texto_youtube_completo = re.sub(r'\s+', ' ', texto_youtube_completo).strip()

    print(f"\n📝 TEXTO COMPLETO DO YOUTUBE:")
    print(f"   {texto_youtube_completo}")
else:
    print("   ❌ YouTube não encontrado! Execute a célula B0 primeiro.")

# ═══════════════════════════════════════════════════════════════════
# 3. TEXTO COMPLETO DO WHISPER
# ═══════════════════════════════════════════════════════════════════

if srt_whisper.exists():
    texto_whisper_completo = " ".join([leg.texto for leg in legendas_whisper])
    texto_whisper_completo = re.sub(r'\s+', ' ', texto_whisper_completo).strip()

    print(f"\n📝 TEXTO COMPLETO DO WHISPER:")
    print(f"   {texto_whisper_completo}")

# ═══════════════════════════════════════════════════════════════════
# 4. RESUMO
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("📊 RESUMO:")
print(f"   Whisper: {len(legendas_whisper) if srt_whisper.exists() else '?'} segmentos")
print(f"   YouTube: {len(legendas_ref) if srt_ref.exists() else '?'} segmentos")
print("═" * 80)

print("\n✅ Diagnóstico concluído!")

════════════════════════════════════════════════════════════════════════════════
🔍 DIAGNÓSTICO - TEXTOS ORIGINAIS
   Oração: pai_nosso
════════════════════════════════════════════════════════════════════════════════

📁 1. WHISPER (FASE 2)
────────────────────────────────────────────────────────────────────────────────
   Segmentos: 7

   [01] 00:00:00,000 → 00:00:04,460
        Pai nosso que estáis no céu, santificados sejam o vosso nome.

   [02] 00:00:05,540 → 00:00:06,880
        Vem a nós o vosso reino.

   [03] 00:00:07,800 → 00:00:11,980
        Seja feita a vossa vontade, assim na Terra como no céu.

   [04] 00:00:12,940 → 00:00:15,020
        O pão nosso de cada dia nos dá hoje.

   [05] 00:00:15,980 → 00:00:17,720
        Perdua aí as nossas ofensas.

   [06] 00:00:18,600 → 00:00:21,400
        Assim como nós perduamos aqui nos tem ofendido.

   [07] 00:00:22,180 → 00:00:26,700
        E não nos deixe cair em tentação, mas livrar-nos do mal a mim.


📁 2. YOUTUBE (REFERÊNCIA - 

In [13]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎯 FASE 3 COMPLETA - Análise e Distribuição                   ║
# ║  Mostra: Whisper original + YouTube original + Resultado       ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
import re

print("=" * 80)
print("🎯 FASE 3 - ANÁLISE COMPLETA")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# ===================================================================
# 1. WHISPER ORIGINAL (timestamps e texto)
# ===================================================================
srt_whisper = Path(f'{config.NOME_ORACAO}_pt_edge.srt')
if not srt_whisper.exists():
    for nome in [f'{config.NOME_ORACAO}_pt_whisper.srt', 'pai_nosso_pt_edge.srt']:
        if Path(nome).exists():
            srt_whisper = Path(nome)
            break

legendas_whisper = ler_srt(srt_whisper)
n_whisper = len(legendas_whisper)

print("\n" + "=" * 80)
print("📁 1. WHISPER ORIGINAL (MESTRE DE TIMESTAMPS)")
print("=" * 80)
print(f"   Total de segmentos: {n_whisper}")
print()
for leg in legendas_whisper:
    print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
    print(f"     {leg.texto}")
    print()

# ===================================================================
# 2. YOUTUBE ORIGINAL (texto oficial com timestamps originais)
# ===================================================================
srt_ref = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
legendas_ref = ler_srt(srt_ref)
n_ref = len(legendas_ref)

print("=" * 80)
print("📁 2. YOUTUBE ORIGINAL (MESTRE DE TEXTO)")
print("=" * 80)
print(f"   Total de segmentos: {n_ref}")
print()
for leg in legendas_ref:
    print(f"[{leg.id:02d}] {leg.texto}")
print()

# ===================================================================
# 3. TEXTO COMPLETO DO YOUTUBE
# ===================================================================
texto_completo = " ".join([leg.texto for leg in legendas_ref])
texto_completo = re.sub(r'\[.*?\]', '', texto_completo)
texto_completo = re.sub(r'M\.?', '', texto_completo)
texto_completo = re.sub(r'METRO', '', texto_completo)
texto_completo = re.sub(r'\s+', ' ', texto_completo).strip()

print("=" * 80)
print("📁 3. TEXTO YOUTUBE LIMPO")
print("=" * 80)
print(f"   {texto_completo}")
print(f"   Total caracteres: {len(texto_completo)}")
print()

# ===================================================================
# 4. DIVIDIR EM FRASES
# ===================================================================
frases = texto_completo.split('. ')
frases = [f.strip() + '.' for f in frases if f.strip()]
n_frases = len(frases)

print("=" * 80)
print("📁 4. FRASES IDENTIFICADAS")
print("=" * 80)
for i, f in enumerate(frases, 1):
    print(f"   Frase {i:02d}: {f}")
print()

# ===================================================================
# 5. AJUSTAR PARA TER EXATAMENTE n_whisper PARTES
# ===================================================================
print("=" * 80)
print("📁 5. AJUSTANDO PARA 7 SEGMENTOS")
print("=" * 80)

if n_frases < n_whisper:
    print(f"⚠️ {n_frases} frases para {n_whisper} segmentos - quebrando frases longas")

    # Quebrar a primeira frase (a mais longa) em partes
    primeira_frase = frases[0]
    # Remover o ponto final temporariamente
    primeira_frase_sem_ponto = primeira_frase.rstrip('.')

    # Dividir por vírgulas
    partes = [p.strip() for p in primeira_frase_sem_ponto.split(',')]

    # Garantir que cada parte termina com a pontuação adequada
    for i, p in enumerate(partes):
        if i < len(partes) - 1:
            partes[i] = p + ','
        else:
            partes[i] = p + '.'

    # Substituir a primeira frase pelas partes
    novas_frases = partes + frases[1:]
    n_novas = len(novas_frases)

    print(f"\n   Após quebra: {n_novas} partes")
    for i, p in enumerate(novas_frases, 1):
        print(f"      Parte {i:02d}: {p}")

    frases = novas_frases
    n_frases = n_novas

# Se ainda tiver menos, duplicar a última (mas sem perder)
if n_frases < n_whisper:
    print(f"\n⚠️ Ainda faltam {n_whisper - n_frases} segmentos - ajustando...")
    while len(frases) < n_whisper:
        frases.append("")
    print(f"   Agora: {len(frases)} partes")

# Se tiver mais, combinar as últimas
if n_frases > n_whisper:
    print(f"\n⚠️ {n_frases} partes para {n_whisper} segmentos - combinando...")
    while len(frases) > n_whisper:
        # Combinar as duas últimas
        ultima = frases.pop()
        penultima = frases.pop()
        frases.append(penultima + " " + ultima)
    print(f"   Agora: {len(frases)} partes")

# ===================================================================
# 6. DISTRIBUIÇÃO FINAL
# ===================================================================
print("\n" + "=" * 80)
print("📁 6. RESULTADO FINAL (7 segmentos com texto do YouTube)")
print("=" * 80)

novas_legendas = []
for i, leg in enumerate(legendas_whisper):
    if i < len(frases):
        texto_parte = frases[i]
    else:
        texto_parte = ""

    # Formatação
    if texto_parte and texto_parte[0].islower():
        texto_parte = texto_parte[0].upper() + texto_parte[1:]

    # Garantir ponto final
    if texto_parte and texto_parte[-1] not in '.!?;:':
        texto_parte += '.'

    leg.texto = texto_parte
    novas_legendas.append(leg)

    print(f"\n[{i+1:02d}] {leg.inicio_str} → {leg.fim_str}")
    print(f"     {texto_parte}")

# ===================================================================
# 7. SALVAR
# ===================================================================
srt_pt = Path(config.NOME_SRT_PT)
salvar_srt(novas_legendas, srt_pt)

print("\n" + "=" * 80)
print("✅ RESULTADO SALVO EM:", srt_pt)
print("=" * 80)

# ===================================================================
# 8. RESUMO PARA ANÁLISE
# ===================================================================
print("\n" + "=" * 80)
print("📊 RESUMO PARA ANÁLISE")
print("=" * 80)
print(f"   Whisper: {n_whisper} segmentos")
print(f"   YouTube original: {n_ref} segmentos")
print(f"   Texto YouTube limpo: {len(texto_completo)} caracteres")
print(f"   Frases identificadas: {len(frases)}")
print(f"   Resultado final: {len(novas_legendas)} segmentos")
print("=" * 80)

🎯 FASE 3 - ANÁLISE COMPLETA
   Oração: pai_nosso

📁 1. WHISPER ORIGINAL (MESTRE DE TIMESTAMPS)
   Total de segmentos: 7

[01] 00:00:00,000 → 00:00:04,460
     Pai nosso que estáis no céu, santificados sejam o vosso nome.

[02] 00:00:05,540 → 00:00:06,880
     Vem a nós o vosso reino.

[03] 00:00:07,800 → 00:00:11,980
     Seja feita a vossa vontade, assim na Terra como no céu.

[04] 00:00:12,940 → 00:00:15,020
     O pão nosso de cada dia nos dá hoje.

[05] 00:00:15,980 → 00:00:17,720
     Perdua aí as nossas ofensas.

[06] 00:00:18,600 → 00:00:21,400
     Assim como nós perduamos aqui nos tem ofendido.

[07] 00:00:22,180 → 00:00:26,700
     E não nos deixe cair em tentação, mas livrar-nos do mal a mim.

📁 2. YOUTUBE ORIGINAL (MESTRE DE TEXTO)
   Total de segmentos: 11

[01] Pai nosso que estais no  céu,
[02] santificado seja o vosso nome, venha a
[03] nós o vosso reino, seja feita a
[04] vossa vontade, assim na terra como no
[05] céu. O pão nosso de cada dia nos
[06] dai hoje. Perdoai

In [14]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 GROQ - DISTRIBUIÇÃO BALANCEADA (CORRIGIDA)                   ║
# ║  Mantém 7 segmentos - anexa "Amém" sem reduzir segmentos        ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
from google.colab import userdata
from openai import OpenAI
import json
import re

print("=" * 80)
print("🤖 GROQ - DISTRIBUIÇÃO BALANCEADA (CORRIGIDA)")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# 1. WHISPER ORIGINAL
srt_whisper = Path(f'{config.NOME_ORACAO}_pt_edge.srt')
if not srt_whisper.exists():
    for nome in [f'{config.NOME_ORACAO}_pt_whisper.srt', 'pai_nosso_pt_edge.srt']:
        if Path(nome).exists():
            srt_whisper = Path(nome)
            break

legendas_whisper = ler_srt(srt_whisper)
n_whisper = len(legendas_whisper)

print("\n" + "=" * 80)
print("📁 1. WHISPER ORIGINAL (MESTRE DE TIMESTAMPS)")
print("=" * 80)
for leg in legendas_whisper:
    print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
    print(f"     {leg.texto}")
    print()

# 2. YOUTUBE ORIGINAL
srt_ref = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
legendas_ref = ler_srt(srt_ref)

print("=" * 80)
print("📁 2. YOUTUBE ORIGINAL (TEXTO OFICIAL)")
print("=" * 80)
for leg in legendas_ref:
    print(f"[{leg.id:02d}] {leg.texto}")
print()

# 3. TEXTO YOUTUBE LIMPO
texto_youtube = " ".join([leg.texto for leg in legendas_ref])
texto_youtube = re.sub(r'\[.*?\]', '', texto_youtube)
texto_youtube = re.sub(r'M\.?', '', texto_youtube)
texto_youtube = re.sub(r'METRO', '', texto_youtube)
texto_youtube = re.sub(r'\s+', ' ', texto_youtube).strip()

print("=" * 80)
print("📁 3. TEXTO YOUTUBE LIMPO")
print("=" * 80)
print(f"   {texto_youtube}")
print(f"   Total palavras: {len(texto_youtube.split())}")
print(f"   Total caracteres: {len(texto_youtube)}")
print()

# 4. PROMPT PARA GROQ
timestamps_texto = "\n".join([f"   {i+1}: {leg.inicio_str} → {leg.fim_str}" for i, leg in enumerate(legendas_whisper)])

PROMPT = f"""Distribua o texto abaixo em EXATAMENTE {n_whisper} partes BALANCEADAS.

TIMESTAMPS (ordem):
{timestamps_texto}

TEXTO:
{texto_youtube}

REGRAS:
- Todas as {n_whisper} partes devem ter texto (nenhuma vazia)
- A última parte deve conter "Amém" junto com o texto anterior
- Distribua com aproximadamente o mesmo número de palavras por parte

Responda APENAS com JSON: {{"partes": ["parte1", "parte2", "parte3", "parte4", "parte5", "parte6", "parte7"]}}"""

# 5. CHAMAR GROQ
print("=" * 80)
print("📁 4. ENVIANDO PARA GROQ")
print("=" * 80)

api_key = userdata.get('GROQ_KEY')
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

try:
    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": f"Você divide textos em {n_whisper} partes balanceadas. Responda apenas com JSON."},
            {"role": "user", "content": PROMPT}
        ],
        temperature=0.3,
        max_tokens=1500,
    )

    resultado = resposta.choices[0].message.content.strip()
    resultado = re.sub(r'```json\n?', '', resultado)
    resultado = re.sub(r'```\n?', '', resultado)

    dados = json.loads(resultado)
    partes = dados.get('partes', [])

    print(f"✅ Resposta recebida: {len(partes)} partes")

except Exception as e:
    print(f"⚠️ Erro: {e}")
    partes = []

# 6. APLICAR DISTRIBUIÇÃO
if len(partes) == n_whisper:
    print("\n" + "=" * 80)
    print("📁 5. RESULTADO FINAL")
    print("=" * 80)

    print("\n📊 DISTRIBUIÇÃO DE PALAVRAS:")
    for i, parte in enumerate(partes):
        n_palavras = len(parte.split())
        print(f"   Segmento {i+1}: {n_palavras} palavras")

    print("\n📝 TEXTO POR SEGMENTO:")
    for i, leg in enumerate(legendas_whisper):
        texto_parte = partes[i]

        # Formatação
        if texto_parte and texto_parte[0].islower():
            texto_parte = texto_parte[0].upper() + texto_parte[1:]
        if texto_parte and texto_parte[-1] not in '.!?;:':
            texto_parte += '.'

        texto_parte = re.sub(r',\.$', '.', texto_parte)

        leg.texto = texto_parte

        print(f"\n[{i+1:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"     {texto_parte}")

    # Salvar (sem reduzir segmentos)
    srt_pt = Path(config.NOME_SRT_PT)
    salvar_srt(legendas_whisper, srt_pt)

    print("\n" + "=" * 80)
    print(f"✅ RESULTADO SALVO: {srt_pt}")
    print(f"   Total de segmentos: {len(legendas_whisper)}")
    print("=" * 80)

    print("\n📊 RESUMO:")
    print(f"   Whisper: {n_whisper} segmentos")
    print(f"   YouTube: {n_ref} segmentos")
    print(f"   Resultado: {len(legendas_whisper)} segmentos")
    print("=" * 80)

else:
    print(f"\n❌ Erro: {len(partes)} partes, esperado {n_whisper}")

🤖 GROQ - DISTRIBUIÇÃO BALANCEADA (CORRIGIDA)
   Oração: pai_nosso

📁 1. WHISPER ORIGINAL (MESTRE DE TIMESTAMPS)
[01] 00:00:00,000 → 00:00:04,460
     Pai nosso que estáis no céu, santificados sejam o vosso nome.

[02] 00:00:05,540 → 00:00:06,880
     Vem a nós o vosso reino.

[03] 00:00:07,800 → 00:00:11,980
     Seja feita a vossa vontade, assim na Terra como no céu.

[04] 00:00:12,940 → 00:00:15,020
     O pão nosso de cada dia nos dá hoje.

[05] 00:00:15,980 → 00:00:17,720
     Perdua aí as nossas ofensas.

[06] 00:00:18,600 → 00:00:21,400
     Assim como nós perduamos aqui nos tem ofendido.

[07] 00:00:22,180 → 00:00:26,700
     E não nos deixe cair em tentação, mas livrar-nos do mal a mim.

📁 2. YOUTUBE ORIGINAL (TEXTO OFICIAL)
[01] Pai nosso que estais no  céu,
[02] santificado seja o vosso nome, venha a
[03] nós o vosso reino, seja feita a
[04] vossa vontade, assim na terra como no
[05] céu. O pão nosso de cada dia nos
[06] dai hoje. Perdoai as nossas
[07] ofensas,
[08] assim co

In [16]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 GROQ - REVISÃO FINAL DA DISTRIBUIÇÃO (GENÉRICA)              ║
# ║  Ajusta o texto para seguir a ESTRUTURA do Whisper               ║
# ║  SEM regras específicas - funciona para QUALQUER oração          ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
from google.colab import userdata
from openai import OpenAI
import json
import re

print("=" * 80)
print("🤖 GROQ - REVISÃO FINAL DA DISTRIBUIÇÃO (GENÉRICA)")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# 1. Carregar timestamps do Whisper (a estrutura que queremos manter)
srt_whisper = Path(f'{config.NOME_ORACAO}_pt_edge.srt')
if not srt_whisper.exists():
    for nome in [f'{config.NOME_ORACAO}_pt_whisper.srt', 'pai_nosso_pt_edge.srt']:
        if Path(nome).exists():
            srt_whisper = Path(nome)
            break

legendas_whisper = ler_srt(srt_whisper)
n_whisper = len(legendas_whisper)

print("\n" + "=" * 80)
print("📁 1. ESTRUTURA DO WHISPER (timestamps e segmentação)")
print("=" * 80)
for leg in legendas_whisper:
    print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
    print(f"     {leg.texto}")
    print()

# 2. Carregar o texto do YouTube (texto correto)
srt_ref = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
legendas_ref = ler_srt(srt_ref)
texto_youtube = " ".join([leg.texto for leg in legendas_ref])
texto_youtube = re.sub(r'\[.*?\]', '', texto_youtube)
texto_youtube = re.sub(r'M\.?', '', texto_youtube)
texto_youtube = re.sub(r'METRO', '', texto_youtube)
texto_youtube = re.sub(r'\s+', ' ', texto_youtube).strip()

print("=" * 80)
print("📁 2. TEXTO DO YOUTUBE (texto correto)")
print("=" * 80)
print(f"   {texto_youtube}")
print(f"   Total palavras: {len(texto_youtube.split())}")
print()

# 3. Carregar o resultado atual
srt_atual = Path(config.NOME_SRT_PT)
if srt_atual.exists():
    legendas_atual = ler_srt(srt_atual)
    print("=" * 80)
    print("📁 3. RESULTADO ATUAL (antes da revisão)")
    print("=" * 80)
    for leg in legendas_atual:
        print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"     {leg.texto}")
        print()
else:
    legendas_atual = legendas_whisper

# 4. PROMPT GENÉRICO (sem regras específicas)
timestamps_texto = "\n".join([f"   Segmento {i+1}: {leg.inicio_str} → {leg.fim_str}" for i, leg in enumerate(legendas_whisper)])

PROMPT = f"""Revisar a distribuição do texto abaixo.

## TIMESTAMPS (devem ser mantidos):
{timestamps_texto}

## TEXTO CORRETO DA ORAÇÃO:
{texto_youtube}

## REGRAS:
1. Mantenha os {n_whisper} segmentos com os timestamps acima
2. Use APENAS o texto fornecido
3. Não adicione nem remova palavras
4. Mantenha a ordem cronológica
5. Distribua o texto de forma balanceada pelos {n_whisper} segmentos

## FORMATO DE RESPOSTA:
{{"segmentos": ["texto1", "texto2", ...]}}"""

# 5. Chamar Groq
print("=" * 80)
print("📁 4. ENVIANDO PARA GROQ")
print("=" * 80)

api_key = userdata.get('GROQ_KEY')
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

try:
    resposta = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": f"Você divide textos em {n_whisper} partes. Responda apenas com JSON."},
            {"role": "user", "content": PROMPT}
        ],
        temperature=0.3,
        max_tokens=1500,
    )

    resultado = resposta.choices[0].message.content.strip()
    resultado = re.sub(r'```json\n?', '', resultado)
    resultado = re.sub(r'```\n?', '', resultado)

    dados = json.loads(resultado)
    segmentos_revisados = dados.get('segmentos', [])

    print(f"✅ Resposta recebida: {len(segmentos_revisados)} segmentos")

except Exception as e:
    print(f"⚠️ Erro: {e}")
    segmentos_revisados = []

# 6. Aplicar revisão
if len(segmentos_revisados) == n_whisper:
    print("\n" + "=" * 80)
    print("📁 5. RESULTADO REVISADO")
    print("=" * 80)

    print("\n📊 DISTRIBUIÇÃO:")
    for i, parte in enumerate(segmentos_revisados):
        n_palavras = len(parte.split())
        print(f"   Segmento {i+1}: {n_palavras} palavras")

    print("\n📝 TEXTO POR SEGMENTO:")
    for i, leg in enumerate(legendas_whisper):
        texto_parte = segmentos_revisados[i]

        if texto_parte and texto_parte[0].islower():
            texto_parte = texto_parte[0].upper() + texto_parte[1:]
        if texto_parte and texto_parte[-1] not in '.!?;:':
            texto_parte += '.'

        leg.texto = texto_parte

        print(f"\n[{i+1:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"     {texto_parte}")

    # Salvar
    srt_pt = Path(config.NOME_SRT_PT)
    salvar_srt(legendas_whisper, srt_pt)

    print("\n" + "=" * 80)
    print(f"✅ RESULTADO SALVO: {srt_pt}")
    print(f"   Total de segmentos: {len(legendas_whisper)}")
    print("=" * 80)

else:
    print(f"\n❌ Erro: {len(segmentos_revisados)} segmentos, esperado {n_whisper}")

🤖 GROQ - REVISÃO FINAL DA DISTRIBUIÇÃO (GENÉRICA)
   Oração: pai_nosso

📁 1. ESTRUTURA DO WHISPER (timestamps e segmentação)
[01] 00:00:00,000 → 00:00:04,460
     Pai nosso que estáis no céu, santificados sejam o vosso nome.

[02] 00:00:05,540 → 00:00:06,880
     Vem a nós o vosso reino.

[03] 00:00:07,800 → 00:00:11,980
     Seja feita a vossa vontade, assim na Terra como no céu.

[04] 00:00:12,940 → 00:00:15,020
     O pão nosso de cada dia nos dá hoje.

[05] 00:00:15,980 → 00:00:17,720
     Perdua aí as nossas ofensas.

[06] 00:00:18,600 → 00:00:21,400
     Assim como nós perduamos aqui nos tem ofendido.

[07] 00:00:22,180 → 00:00:26,700
     E não nos deixe cair em tentação, mas livrar-nos do mal a mim.

📁 2. TEXTO DO YOUTUBE (texto correto)
   Pai nosso que estais no céu, santificado seja o vosso nome, venha a nós o vosso reino, seja feita a vossa vontade, assim na terra como no céu. O pão nosso de cada dia nos dai hoje. Perdoai as nossas ofensas, assim como nós perdoamos a quem n

In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 AJUSTE FINAL DE PONTUAÇÃO - Remover vírgulas extras          ║
# ║  Executar APÓS a FASE 3 com Groq                                ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
import re

print("=" * 80)
print("🔧 AJUSTE FINAL - CORRIGINDO PONTUAÇÃO")
print("=" * 80)

# Carregar o SRT gerado pelo Groq
srt_pt = Path(config.NOME_SRT_PT)
legendas = ler_srt(srt_pt)

print("\n📝 ANTES:")
for leg in legendas:
    print(f"[{leg.id}] {leg.texto}")

# Correções
for leg in legendas:
    # Remover ",." no final
    leg.texto = re.sub(r',\.$', '.', leg.texto)
    leg.texto = re.sub(r',\.', '.', leg.texto)

    # Remover espaços antes de pontuação
    leg.texto = re.sub(r'\s+\.', '.', leg.texto)

    # Garantir que não tem vírgula no final
    if leg.texto.endswith(','):
        leg.texto = leg.texto[:-1] + '.'

# Salvar
salvar_srt(legendas, srt_pt)

print("\n📝 DEPOIS:")
for leg in legendas:
    print(f"[{leg.id}] {leg.texto}")

print("\n" + "=" * 80)
print("✅ PONTUAÇÃO CORRIGIDA!")
print("=" * 80)

🔧 AJUSTE FINAL - CORRIGINDO PONTUAÇÃO

📝 ANTES:
[1] Pai nosso que estais no céu, santificado seja o vosso nome,.
[2] Venha a nós o vosso reino, seja feita a vossa vontade,.
[3] Assim na terra como no céu. O pão nosso de cada dia.
[4] Nos dai hoje. Perdoai as nossas ofensas,.
[5] Assim como nós perdoamos a quem nos tem ofendido.
[6] E não nos deixeis cair em tentação,.
[7] Mas livrai-nos do mal. Amém.

📝 DEPOIS:
[1] Pai nosso que estais no céu, santificado seja o vosso nome.
[2] Venha a nós o vosso reino, seja feita a vossa vontade.
[3] Assim na terra como no céu. O pão nosso de cada dia.
[4] Nos dai hoje. Perdoai as nossas ofensas.
[5] Assim como nós perdoamos a quem nos tem ofendido.
[6] E não nos deixeis cair em tentação.
[7] Mas livrai-nos do mal. Amém.

✅ PONTUAÇÃO CORRIGIDA!


In [18]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PADRONIZAR IDIOMAS - 7 segmentos (mesmo do Whisper/PT)         ║
# ║  Com EXIBIÇÃO COMPLETA para análise e comparação                ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt, sincronizar_timestamps
import re

print("=" * 80)
print("📝 PADRONIZANDO IDIOMAS PARA 7 SEGMENTOS")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# 1. Carregar o PT padrão (7 segmentos - timestamps do Whisper)
srt_pt = Path(config.NOME_SRT_PT)
if not srt_pt.exists():
    raise FileNotFoundError("❌ PT corrigido não encontrado! Execute a FASE 3 primeiro.")

legendas_padrao = ler_srt(srt_pt)
n_padrao = len(legendas_padrao)

print("\n" + "=" * 80)
print("📁 1. PT PADRÃO (REFERÊNCIA DE TIMESTAMPS E TEXTO)")
print("=" * 80)
print(f"   Total de segmentos: {n_padrao}")
print()
for leg in legendas_padrao:
    print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
    print(f"     {leg.texto}")
    print()

# 2. Processar cada idioma
pipeline.legendas_idiomas = {"pt": legendas_padrao}

for lang in ["en", "es", "fr"]:
    srt_original = Path(config.nome_srt(lang))

    if not srt_original.exists():
        print(f"\n⚠️ {lang.upper()} não encontrado! Usando PT como fallback.")
        pipeline.legendas_idiomas[lang] = legendas_padrao
        continue

    # Carregar SRT original do YouTube
    legendas_original = ler_srt(srt_original)

    print("\n" + "=" * 80)
    print(f"📁 2. {lang.upper()} ORIGINAL (YouTube)")
    print("=" * 80)
    print(f"   Total de segmentos: {len(legendas_original)}")
    print()
    for leg in legendas_original:
        print(f"[{leg.id:02d}] {leg.texto}")
    print()

    # Juntar TODO o texto do idioma
    texto_completo = " ".join([leg.texto for leg in legendas_original])

    # Limpeza genérica de artefatos
    texto_completo = re.sub(r'\[.*?\]', '', texto_completo)
    texto_completo = re.sub(r'M\.?', '', texto_completo)
    texto_completo = re.sub(r'METRO', '', texto_completo)
    texto_completo = re.sub(r'ETRO', '', texto_completo)
    texto_completo = re.sub(r'\s+', ' ', texto_completo).strip()

    print(f"📝 TEXTO {lang.upper()} LIMPO:")
    print(f"   {texto_completo}")
    print(f"   Total caracteres: {len(texto_completo)}")
    print(f"   Total palavras: {len(texto_completo.split())}")
    print()

    # Dividir o texto em N_PADRAO partes iguais (baseado em caracteres)
    total_chars = len(texto_completo)
    chars_por_segmento = total_chars / n_padrao

    novas_legendas = []
    pos = 0

    for i, leg_padrao in enumerate(legendas_padrao):
        fim_pos = int(pos + chars_por_segmento)

        # Ajustar para não cortar palavras no meio
        if fim_pos < total_chars:
            while fim_pos < total_chars and texto_completo[fim_pos] != ' ':
                fim_pos += 1

        texto_parte = texto_completo[pos:fim_pos].strip()

        # Último segmento pega o resto
        if i == n_padrao - 1:
            texto_parte = texto_completo[pos:].strip()

        # Garantir pontuação no final
        if texto_parte and texto_parte[-1] not in '.!?;:':
            texto_parte += '.'

        # Primeira letra maiúscula
        if texto_parte and texto_parte[0].islower():
            texto_parte = texto_parte[0].upper() + texto_parte[1:]

        # Corrigir vírgulas extras
        texto_parte = re.sub(r',\.$', '.', texto_parte)
        texto_parte = re.sub(r',\.', '.', texto_parte)

        # Criar nova legenda com timestamp do PT
        from models import Legenda
        nova_legenda = Legenda(
            id=i + 1,
            inicio_ms=leg_padrao.inicio_ms,
            fim_ms=leg_padrao.fim_ms,
            texto=texto_parte,
            palavras=[]
        )
        novas_legendas.append(nova_legenda)

        pos = fim_pos

    # Salvar SRT padronizado
    srt_padrao = Path(f'{config.NOME_ORACAO}_{lang}_padrao.srt')
    salvar_srt(novas_legendas, srt_padrao)

    # Atualizar no pipeline
    pipeline.legendas_idiomas[lang] = novas_legendas

    print("=" * 80)
    print(f"📁 3. {lang.upper()} PADRONIZADO (7 segmentos)")
    print("=" * 80)
    print(f"   Total de segmentos: {len(novas_legendas)}")
    print()

    # Mostrar estatísticas
    print("📊 DISTRIBUIÇÃO DE PALAVRAS:")
    for j, leg in enumerate(novas_legendas):
        n_palavras = len(leg.texto.split())
        print(f"   Segmento {j+1}: {n_palavras} palavras")
    print()

    print("📝 TEXTO COMPLETO POR SEGMENTO:")
    for leg in novas_legendas:
        print(f"[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"     {leg.texto}")
        print()

# ═══════════════════════════════════════════════════════════════════
# VERIFICAÇÃO FINAL COMPARATIVA
# ═══════════════════════════════════════════════════════════════════

print("\n" + "=" * 80)
print("📊 VERIFICAÇÃO FINAL - COMPARAÇÃO ENTRE IDIOMAS")
print("=" * 80)
print(f"{'Idioma':<6} {'Segmentos':<10} {'Primeira legenda':<40} {'Última legenda':<40}")
print("-" * 80)

for lang in ["pt", "en", "es", "fr"]:
    if lang in pipeline.legendas_idiomas:
        n_seg = len(pipeline.legendas_idiomas[lang])
        primeira = pipeline.legendas_idiomas[lang][0].texto[:50]
        ultima = pipeline.legendas_idiomas[lang][-1].texto[:50]
        print(f"{lang.upper():<6} {n_seg:<10} {primeira:<40} {ultima:<40}")
    else:
        print(f"{lang.upper():<6} {'não carregado':<10} {'-':<40} {'-':<40}")

print("\n" + "=" * 80)
print("✅ TODOS OS IDIOMAS PADRONIZADOS COM 7 SEGMENTOS!")
print("=" * 80)

print("\n📌 RESUMO:")
print(f"   PT: {len(pipeline.legendas_idiomas['pt'])} segmentos (referência)")
print(f"   EN: {len(pipeline.legendas_idiomas['en'])} segmentos")
print(f"   ES: {len(pipeline.legendas_idiomas['es'])} segmentos")
print(f"   FR: {len(pipeline.legendas_idiomas['fr'])} segmentos")
print("=" * 80)

print("\n✅ Agora execute a FASE 5A (Classificação Morfológica)!")

📝 PADRONIZANDO IDIOMAS PARA 7 SEGMENTOS
   Oração: pai_nosso

📁 1. PT PADRÃO (REFERÊNCIA DE TIMESTAMPS E TEXTO)
   Total de segmentos: 7

[01] 00:00:00,000 → 00:00:04,460
     Pai nosso que estais no céu, santificado seja o vosso nome.

[02] 00:00:05,540 → 00:00:06,880
     Venha a nós o vosso reino, seja feita a vossa vontade.

[03] 00:00:07,800 → 00:00:11,980
     Assim na terra como no céu. O pão nosso de cada dia.

[04] 00:00:12,940 → 00:00:15,020
     Nos dai hoje. Perdoai as nossas ofensas.

[05] 00:00:15,980 → 00:00:17,720
     Assim como nós perdoamos a quem nos tem ofendido.

[06] 00:00:18,600 → 00:00:21,400
     E não nos deixeis cair em tentação.

[07] 00:00:22,180 → 00:00:26,700
     Mas livrai-nos do mal. Amém.


📁 2. EN ORIGINAL (YouTube)
   Total de segmentos: 11

[01] Our Father who art in heaven,
[02] hallowed be thy name,
[03] thy kingdom come,
[04] thy will be done, on earth as it is in
[05] heaven.  Give us this day our daily bread
[06] .  Forgive us our
[07] trespa

In [19]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 CORREÇÃO DE EN/ES/FR - USAR RESPOSTA COMPLETA DO GROQ       ║
# ║  O Groq retorna os 7 segmentos prontos. Não recortar!           ║
# ║  PT corrigido é a referência para estrutura                     ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
from google.colab import userdata
from openai import OpenAI
import json
import re

print("=" * 80)
print("🤖 CORREÇÃO DE EN/ES/FR")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# 1. Carregar PT CORRIGIDO (referência de estrutura)
srt_pt = Path(config.NOME_SRT_PT)
legendas_pt = ler_srt(srt_pt)

print("\n" + "=" * 80)
print("📁 PT CORRIGIDO (REFERÊNCIA DE ESTRUTURA)")
print("=" * 80)
for leg in legendas_pt:
    print(f"[{leg.id:02d}] {leg.texto}")
print()

# Construir o texto do PT com timestamps para referência
pt_com_timestamps = "\n".join([f"{leg.id}: {leg.texto}" for leg in legendas_pt])

api_key = userdata.get('GROQ_KEY')
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

for lang in ["en", "es", "fr"]:
    if lang not in pipeline.legendas_idiomas:
        continue

    legendas = pipeline.legendas_idiomas[lang]

    # Juntar texto atual do idioma
    texto_atual = " ".join([leg.texto for leg in legendas])

    # PROMPT pedindo para o Groq retornar os 7 segmentos prontos
    PROMPT = f"""O texto em PORTUGUÊS abaixo está PERFEITO e serve como REFERÊNCIA de estrutura (7 segmentos):

{pt_com_timestamps}

Agora, corrija o texto em {lang.upper()} abaixo, mantendo EXATAMENTE 7 segmentos com a MESMA ESTRUTURA do português.

TEXTO EM {lang.upper()} PARA CORRIGIR:
{texto_atual}

REGRAS:
- Retorne EXATAMENTE 7 segmentos
- Cada segmento deve corresponder ao segmento do português
- Corrija apenas erros, não reescreva
- Não adicione texto extra

Responda APENAS com JSON no formato:
{{"segmentos": ["texto1", "texto2", "texto3", "texto4", "texto5", "texto6", "texto7"]}}"""

    print(f"\n🤖 Corrigindo {lang.upper()}...")

    try:
        resposta = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": f"Você corrige textos em {lang.upper()} mantendo exatamente 7 segmentos. Responda apenas com JSON."},
                {"role": "user", "content": PROMPT}
            ],
            temperature=0.2,
            max_tokens=1500,
        )

        resultado = resposta.choices[0].message.content.strip()
        resultado = re.sub(r'```json\n?', '', resultado)
        resultado = re.sub(r'```\n?', '', resultado)

        dados = json.loads(resultado)
        segmentos = dados.get('segmentos', [])

        if len(segmentos) == 7:
            print(f"\n✅ {lang.upper()} CORRIGIDO:")

            # Criar novas legendas com os segmentos do Groq
            novas_legendas = []
            for i, leg_pt in enumerate(legendas_pt):
                texto = segmentos[i]

                # Formatação básica
                if texto and texto[0].islower():
                    texto = texto[0].upper() + texto[1:]
                if texto and texto[-1] not in '.!?;:':
                    texto += '.'

                from models import Legenda
                nova_legenda = Legenda(
                    id=i+1,
                    inicio_ms=leg_pt.inicio_ms,
                    fim_ms=leg_pt.fim_ms,
                    texto=texto,
                    palavras=[]
                )
                novas_legendas.append(nova_legenda)
                print(f"   [{i+1:02d}] {texto}")

            # Salvar
            pipeline.legendas_idiomas[lang] = novas_legendas
            srt_path = Path(config.nome_srt(lang))
            salvar_srt(novas_legendas, srt_path)

        else:
            print(f"\n⚠️ {lang.upper()}: retornou {len(segmentos)} segmentos, esperado 7")

    except Exception as e:
        print(f"   ⚠️ Erro: {e}")

print("\n" + "=" * 80)
print("✅ CORREÇÃO CONCLUÍDA!")
print("=" * 80)

🤖 CORREÇÃO DE EN/ES/FR
   Oração: pai_nosso

📁 PT CORRIGIDO (REFERÊNCIA DE ESTRUTURA)
[01] Pai nosso que estais no céu, santificado seja o vosso nome.
[02] Venha a nós o vosso reino, seja feita a vossa vontade.
[03] Assim na terra como no céu. O pão nosso de cada dia.
[04] Nos dai hoje. Perdoai as nossas ofensas.
[05] Assim como nós perdoamos a quem nos tem ofendido.
[06] E não nos deixeis cair em tentação.
[07] Mas livrai-nos do mal. Amém.


🤖 Corrigindo EN...

✅ EN CORRIGIDO:
   [01] Our Father who art in heaven, hallowed be thy name.
   [02] Thy kingdom come, thy will be done.
   [03] On earth as it is in heaven.
   [04] Give us this day our daily bread.
   [05] Forgive us our trespasses, as we forgive those who trespass against us.
   [06] And lead us not into temptation.
   [07] Deliver us from evil. Amen.

🤖 Corrigindo ES...

✅ ES CORRIGIDO:
   [01] Padre nuestro que estás en los cielos, santificado sea tu nombre.
   [02] Ven a nosotros tu reino, hágase tu voluntad.
   [03] Así e

In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 CORREÇÃO FINAL - Palavras e Pontuação (IA)                   ║
# ║  Usa Groq/Mistral para corrigir pequenos erros nos 4 idiomas    ║
# ║  Mantém os 7 segmentos e timestamps do PT                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
from google.colab import userdata
from openai import OpenAI
import json
import re

print("=" * 80)
print("🤖 CORREÇÃO FINAL - PALAVRAS E PONTUAÇÃO")
print(f"   Oração: {config.NOME_ORACAO}")
print("=" * 80)

# Carregar PT como referência (estrutura perfeita)
srt_pt = Path(config.NOME_SRT_PT)
legendas_pt = ler_srt(srt_pt)

print("\n📁 PT (REFERÊNCIA) - ESTRUTURA CORRETA:")
for leg in legendas_pt:
    print(f"   [{leg.id:02d}] {leg.texto}")
print()

# Texto do PT para referência
texto_pt_referencia = "\n".join([f"{i+1}: {leg.texto}" for i, leg in enumerate(legendas_pt)])

api_key = userdata.get('GROQ_KEY')
client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")

for lang in ["en", "es", "fr"]:
    if lang not in pipeline.legendas_idiomas:
        continue

    legendas = pipeline.legendas_idiomas[lang]

    print(f"\n📥 {lang.upper()} ANTES DA CORREÇÃO:")
    for leg in legendas:
        print(f"   [{leg.id:02d}] {leg.texto}")

    # Construir o texto atual do idioma (separado por linhas)
    texto_atual = "\n".join([f"{i+1}: {leg.texto}" for i, leg in enumerate(legendas)])

    # PROMPT para correção de palavras e pontuação
    PROMPT = f"""Corrija o texto em {lang.upper()} abaixo.

## REFERÊNCIA (PORTUGUÊS - estrutura correta):
{texto_pt_referencia}

## TEXTO EM {lang.upper()} PARA CORRIGIR:
{texto_atual}

## TAREFA:
1. Corrija palavras erradas
2. Corrija pontuação (vírgulas, pontos, espaços)
3. Remova "," isolado
4. Mantenha exatamente 7 linhas (uma para cada segmento)
5. Mantenha o mesmo significado

Responda APENAS com as 7 linhas corrigidas, uma por linha, sem numeração, sem explicações."""

    print(f"\n🤖 Corrigindo {lang.upper()}...")

    try:
        resposta = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": f"Você corrige textos em {lang.upper()}. Responda apenas com as 7 linhas corrigidas, uma por linha."},
                {"role": "user", "content": PROMPT}
            ],
            temperature=0.1,
            max_tokens=1500,
        )

        texto_corrigido = resposta.choices[0].message.content.strip()
        linhas = [l.strip() for l in texto_corrigido.split('\n') if l.strip()]

        if len(linhas) == 7:
            print(f"\n✅ {lang.upper()} CORRIGIDO:")

            novas_legendas = []
            for i, leg_pt in enumerate(legendas_pt):
                texto = linhas[i]

                # Formatação básica
                if texto and texto[0].islower():
                    texto = texto[0].upper() + texto[1:]
                if texto and texto[-1] not in '.!?;:':
                    texto += '.'
                # Remover ",." no final
                texto = re.sub(r',\.$', '.', texto)

                from models import Legenda
                nova_legenda = Legenda(
                    id=i+1,
                    inicio_ms=leg_pt.inicio_ms,
                    fim_ms=leg_pt.fim_ms,
                    texto=texto,
                    palavras=[]
                )
                novas_legendas.append(nova_legenda)
                print(f"   [{i+1:02d}] {texto}")

            pipeline.legendas_idiomas[lang] = novas_legendas
            srt_path = Path(config.nome_srt(lang))
            salvar_srt(novas_legendas, srt_path)
        else:
            print(f"   ⚠️ Retornou {len(linhas)} linhas, esperado 7")

    except Exception as e:
        print(f"   ⚠️ Erro: {e}")

print("\n" + "=" * 80)
print("✅ CORREÇÃO FINAL CONCLUÍDA!")
print("=" * 80)

# Mostrar resultado final de todos os idiomas
print("\n" + "=" * 80)
print("📋 RESULTADO FINAL - TODOS OS IDIOMAS:")
print("=" * 80)

for lang in ["pt", "en", "es", "fr"]:
    if lang in pipeline.legendas_idiomas:
        print(f"\n{lang.upper()}:")
        for leg in pipeline.legendas_idiomas[lang]:
            print(f"   [{leg.id:02d}] {leg.texto}")

print("\n" + "=" * 80)
print("✅ PRONTO PARA FASE 5A (Classificação Morfológica)!")
print("=" * 80)

🤖 CORREÇÃO FINAL - PALAVRAS E PONTUAÇÃO
   Oração: pai_nosso

📁 PT (REFERÊNCIA) - ESTRUTURA CORRETA:
   [01] Pai nosso que estais no céu, santificado seja o vosso nome.
   [02] Venha a nós o vosso reino, seja feita a vossa vontade.
   [03] Assim na terra como no céu. O pão nosso de cada dia.
   [04] Nos dai hoje. Perdoai as nossas ofensas.
   [05] Assim como nós perdoamos a quem nos tem ofendido.
   [06] E não nos deixeis cair em tentação.
   [07] Mas livrai-nos do mal. Amém.


📥 EN ANTES DA CORREÇÃO:
   [01] Our Father who art in heaven, hallowed be thy name.
   [02] Thy kingdom come, thy will be done.
   [03] On earth as it is in heaven.
   [04] Give us this day our daily bread.
   [05] Forgive us our trespasses, as we forgive those who trespass against us.
   [06] And lead us not into temptation.
   [07] Deliver us from evil. Amen.

🤖 Corrigindo EN...

✅ EN CORRIGIDO:
   [01] Our Father who art in heaven hallowed be thy name.
   [02] Thy kingdom come thy will be done.
   [03] On ear

In [21]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 AJUSTE FINAL - Pontuação (GENÉRICO)                         ║
# ║  Garante que todos os segmentos terminam com pontuação          ║
# ║  Funciona para qualquer oração                                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
import re

print("=" * 80)
print("🔧 AJUSTE FINAL - PONTUAÇÃO")
print("=" * 80)

for lang in ["pt", "en", "es", "fr"]:
    if lang not in pipeline.legendas_idiomas:
        continue

    legendas = pipeline.legendas_idiomas[lang]

    print(f"\n📥 {lang.upper()}:")

    for leg in legendas:
        original = leg.texto

        # Garantir ponto final no final de cada legenda
        if original and original[-1] not in '.!?;:':
            leg.texto = original + '.'

        # Remover vírgula antes de ponto
        leg.texto = re.sub(r',\.$', '.', leg.texto)

        print(f"   [{leg.id:02d}] {leg.texto}")

    # Salvar
    srt_path = Path(config.nome_srt(lang))
    salvar_srt(legendas, srt_path)

print("\n" + "=" * 80)
print("✅ AJUSTE CONCLUÍDO!")
print("=" * 80)

🔧 AJUSTE FINAL - PONTUAÇÃO

📥 PT:
   [01] Pai nosso que estais no céu, santificado seja o vosso nome.
   [02] Venha a nós o vosso reino, seja feita a vossa vontade.
   [03] Assim na terra como no céu. O pão nosso de cada dia.
   [04] Nos dai hoje. Perdoai as nossas ofensas.
   [05] Assim como nós perdoamos a quem nos tem ofendido.
   [06] E não nos deixeis cair em tentação.
   [07] Mas livrai-nos do mal. Amém.

📥 EN:
   [01] Our Father who art in heaven hallowed be thy name.
   [02] Thy kingdom come thy will be done.
   [03] On earth as it is in heaven.
   [04] Give us this day our daily bread.
   [05] Forgive us our trespasses as we forgive those who trespass against us.
   [06] And lead us not into temptation.
   [07] Deliver us from evil Amen.

📥 ES:
   [01] Padre nuestro que estás en los cielos, santificado sea tu nombre.
   [02] Ven a nosotros tu reino, hágase tu voluntad.
   [03] Así en la tierra como en el cielo. Nuestro pan de cada día dánoslo hoy.
   [04] Perdona nuestras ofen

In [22]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 ELIMINAR GAPS DE TODOS OS IDIOMAS (pós-processamento)        ║
# ║  Remove intervalos vazios entre legendas em PT, EN, ES, FR       ║
# ║  Mantém a sincronia entre todos os idiomas                       ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt, eliminar_gaps

print("=" * 80)
print("🔧 ELIMINANDO GAPS DE TODOS OS IDIOMAS")
print("=" * 80)

for lang in ["pt", "en", "es", "fr"]:
    srt_path = Path(config.nome_srt(lang))

    if not srt_path.exists():
        print(f"\n⚠️ {lang.upper()} não encontrado!")
        continue

    # Carregar SRT
    legendas = ler_srt(srt_path)

    print(f"\n📥 {lang.upper()} ANTES:")
    for leg in legendas:
        print(f"   [{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")

    # Eliminar gaps
    legendas = eliminar_gaps(legendas)

    print(f"\n✅ {lang.upper()} DEPOIS:")
    for leg in legendas:
        print(f"   [{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")

    # Salvar
    salvar_srt(legendas, srt_path)

    # Atualizar pipeline
    if lang in pipeline.legendas_idiomas:
        pipeline.legendas_idiomas[lang] = legendas

print("\n" + "=" * 80)
print("✅ GAPS ELIMINADOS EM TODOS OS IDIOMAS!")
print("=" * 80)

# Verificar sincronia final
print("\n📊 VERIFICANDO SINCRONIA FINAL:")
for lang in ["pt", "en", "es", "fr"]:
    srt_path = Path(config.nome_srt(lang))
    if srt_path.exists():
        legendas = ler_srt(srt_path)
        print(f"\n{lang.upper()}:")
        for leg in legendas:
            print(f"   [{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")

🔧 ELIMINANDO GAPS DE TODOS OS IDIOMAS

📥 PT ANTES:
   [01] 00:00:00,000 → 00:00:04,460
   [02] 00:00:05,540 → 00:00:06,880
   [03] 00:00:07,800 → 00:00:11,980
   [04] 00:00:12,940 → 00:00:15,020
   [05] 00:00:15,980 → 00:00:17,720
   [06] 00:00:18,600 → 00:00:21,400
   [07] 00:00:22,180 → 00:00:26,700

✅ PT DEPOIS:
   [01] 00:00:00,000 → 00:00:05,539
   [02] 00:00:05,540 → 00:00:07,799
   [03] 00:00:07,800 → 00:00:12,939
   [04] 00:00:12,940 → 00:00:15,979
   [05] 00:00:15,980 → 00:00:18,599
   [06] 00:00:18,600 → 00:00:22,179
   [07] 00:00:22,180 → 00:00:26,700

📥 EN ANTES:
   [01] 00:00:00,000 → 00:00:04,460
   [02] 00:00:05,540 → 00:00:06,880
   [03] 00:00:07,800 → 00:00:11,980
   [04] 00:00:12,940 → 00:00:15,020
   [05] 00:00:15,980 → 00:00:17,720
   [06] 00:00:18,600 → 00:00:21,400
   [07] 00:00:22,180 → 00:00:26,700

✅ EN DEPOIS:
   [01] 00:00:00,000 → 00:00:05,539
   [02] 00:00:05,540 → 00:00:07,799
   [03] 00:00:07,800 → 00:00:12,939
   [04] 00:00:12,940 → 00:00:15,979
   [05] 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 FORÇAR NOVA CLASSIFICAÇÃO (ignorar cache e JSONs existentes)║
# ║  Executar ANTES da FASE 5A                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import shutil
from pathlib import Path

print("=" * 70)
print("🔧 LIMPANDO CACHES E JSONS EXISTENTES")
print("=" * 70)

# 1. Limpar cache de classificação
cache_dir = Path('/content/cache_classificacao')
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print("✅ Cache de classificação removido")

# 2. Limpar JSONs locais
for lang in ["pt", "en", "es", "fr"]:
    json_path = Path(f'classificacao_{config.NOME_ORACAO}_{lang}.json')
    if json_path.exists():
        json_path.unlink()
        print(f"✅ {json_path.name} removido")

# 3. Limpar JSONs no Drive (pasta brutos)
pasta_brutos = Path('/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/brutos/pai_nosso')
if pasta_brutos.exists():
    for json_path in pasta_brutos.glob('*.json'):
        json_path.unlink()
        print(f"✅ brutos/{json_path.name} removido")

# 4. Limpar JSONs na pasta de correcoes (se existir)
pasta_correcoes = Path('/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/pai_nosso')
if pasta_correcoes.exists():
    for json_path in pasta_correcoes.glob('*.json'):
        json_path.unlink()
        print(f"✅ correcoes/{json_path.name} removido")

# 5. Resetar checkpoint
pipeline._cp.reiniciar_de('classificacoes_feitas')
print("✅ Checkpoint resetado")

# 6. Limpar cache do Classifier
pipeline._clf.invalidar_cache()
print("✅ Cache do Classifier invalidado")

print("\n" + "=" * 70)
print("✅ LIMPEZA CONCLUÍDA! Agora execute a FASE 5A novamente.")
print("=" * 70)

▶ FASE 5A — Classificação Morfológica (Groq → Mistral) O que faz:

Classifica todos os idiomas via API (Groq ou Mistral)

Salva JSONs brutos localmente e faz backup no Drive (pasta brutos/)

⚠️ Após esta célula, execute a célula "PACOTE DE REVISÃO"

In [23]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🤖 FASE 5A - VERSÃO CORRIGIDA COM MISTRAL PRIMEIRO              ║
# ║  Faz a classificação chamando cada API diretamente              ║
# ║  Evita o groq_client.py que está com problema                   ║
# ║                                                                    ║
# ║  CORREÇÕES NESTA VERSÃO:                                          ║
# ║   - Prompt agora inclui a lista REAL de classes válidas          ║
# ║     (lida de constants.CORES_HTML)                               ║
# ║   - Regras linguísticas do projeto incluídas no prompt           ║
# ║   - Cada palavra passa por normalizar_classe() antes de salvar   ║
# ║   - Tokens de pontuação (",", ".", "-") são descartados          ║
# ║   - Parsing de JSON não destrói apóstrofos (aujourd'hui etc.)    ║
# ╚══════════════════════════════════════════════════════════════════╝

import time
import json
import re
from datetime import datetime
from google.colab import userdata
from openai import OpenAI
from pathlib import Path
from models import Palavra
from constants import CORES_HTML, PROMPT_SISTEMA_CLASSIFICACAO, normalizar_classe

print("=" * 70)
print("🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA")
print("   Chamando APIs diretamente (Mistral → Groq)")
print("=" * 70)

# Configurações
DELAY_ENTRE_LEGENDAS = 2
DELAY_ENTRE_IDIOMAS = 5

# Tokens que não são palavras e devem ser descartados
TOKENS_IGNORADOS = {",", ".", "-", "!", "?", ";", ":", "...", "—", "–", '"', "'"}

print(f"⏱️  Delay entre legendas: {DELAY_ENTRE_LEGENDAS}s")
print(f"⏱️  Delay entre idiomas: {DELAY_ENTRE_IDIOMAS}s")
print("=" * 70)

# Configurar APIs (Mistral primeiro)
APIS = [
    {
        'nome': 'Mistral',
        'client': OpenAI(api_key=userdata.get('MISTRAL_KEY'), base_url='https://api.mistral.ai/v1'),
        'modelo': 'mistral-small-latest',
    },
    {
        'nome': 'Groq',
        'client': OpenAI(api_key=userdata.get('GROQ_KEY'), base_url='https://api.groq.com/openai/v1'),
        'modelo': 'llama-3.3-70b-versatile',
    },
]

# ── Monta a lista de classes válidas a partir de constants.CORES_HTML ────────
CLASSES_VALIDAS = sorted(CORES_HTML.keys())
LISTA_CLASSES_MD = "\n".join(f"- {c}" for c in CLASSES_VALIDAS)

# ── Prompt do sistema: usa o PROMPT_SISTEMA_CLASSIFICACAO de constants.py
#    + injeta a lista real de classes válidas ─────────────────────────────────
SYSTEM_PROMPT = (
    PROMPT_SISTEMA_CLASSIFICACAO
    + "\n\n## CLASSES VÁLIDAS (use SOMENTE estas, exatamente como escritas):\n"
    + LISTA_CLASSES_MD
    + "\n\nIMPORTANTE:"
    + "\n- NÃO inclua sinais de pontuação (',', '.', '-', '!', '?', ';', ':') na lista de palavras."
    + "\n- NÃO invente classes que não estejam na lista acima."
    + "\n- Cada token do texto deve aparecer no máximo uma vez na ordem em que aparece."
)


# ── Mapeamento extra (classes genéricas em ES/FR que MAPEAMENTO_CLASSES de
#    constants.py ainda não cobre). Aplicado ANTES de normalizar_classe.
MAPEAMENTO_EXTRA: dict[str, str] = {
    # Francês
    "nom": "substantivo_masculino_singular",
    "verbe": "verbo_presente",
    "pronom": "pronome_pessoal",
    "adjectif": "adjetivo_normal",
    "adverbe": "advérbio_normal",
    "préposition": "preposicao",
    "conjonction": "conjuncao",
    "déterminant": "artigo_definido",
    "adjectif possessif": "pronome_possessivo_singular",
    # Espanhol
    "sustantivo": "substantivo_masculino_singular",
    "pronombre": "pronome_pessoal",
    "adverbio": "advérbio_normal",
    "preposición": "preposicao",
    "conjunción": "conjuncao",
    "interjección": "interjeicao",
    # Genéricos PT/EN adicionais
    "substantivo masculino singular": "substantivo_masculino_singular",
    "substantivo feminino singular": "substantivo_feminino_singular",
    "substantivo masculino plural": "substantivo_masculino_plural",
    "substantivo feminino plural": "substantivo_feminino_plural",
    "pronome possessivo": "pronome_possessivo_singular",
    "pronome pessoal": "pronome_pessoal",
    "pronome relativo": "pronome_relativo",
    "verbo": "verbo_presente",
    "interjeição": "interjeicao",
}


def classificar_legenda(texto, lang):
    """Tenta classificar com Mistral, se falhar usa Groq"""

    user_prompt = f"""Idioma: {lang}
Texto: "{texto}"

Classifique cada palavra do texto acima. Responda APENAS com JSON no formato:
{{"palavras": [{{"texto": "palavra", "classe": "classe"}}]}}"""

    for api in APIS:
        try:
            print(f"      🔄 Tentando: {api['nome']}...")

            resposta = api['client'].chat.completions.create(
                model=api['modelo'],
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2,
                max_tokens=1000,
            )

            resultado = resposta.choices[0].message.content.strip()

            # Remove cercas de código markdown, se houver
            resultado = re.sub(r'^```(?:json)?\s*', '', resultado)
            resultado = re.sub(r'\s*```$', '', resultado)
            resultado = resultado.strip()

            dados = json.loads(resultado)

            if 'palavras' in dados:
                print(f"      ✅ {api['nome']} respondeu!")
                palavras = []
                for p in dados['palavras']:
                    texto_palavra = (p.get('texto') or "").strip()
                    if not texto_palavra:
                        continue
                    if texto_palavra in TOKENS_IGNORADOS:
                        continue

                    classe_bruta = p.get('classe', 'substantivo_masculino_singular')

                    # 1. Mapeamento extra (classes genéricas FR/ES não cobertas
                    #    por constants.MAPEAMENTO_CLASSES)
                    classe_bruta_norm = MAPEAMENTO_EXTRA.get(
                        classe_bruta.strip().lower(), classe_bruta
                    )

                    # 2. Normaliza usando as regras já consolidadas em constants.py
                    #    (correções globais, por idioma, e mapeamento ingles->pt)
                    classe = normalizar_classe(classe_bruta_norm, texto_palavra, lang)

                    # Fallback final: se ainda não for uma classe válida (com cor
                    # definida), usa substantivo_masculino_singular e avisa
                    if classe not in CORES_HTML:
                        print(f"      ⚠️ Classe '{classe_bruta}' (→ '{classe}') inválida "
                              f"para '{texto_palavra}' — usando fallback")
                        classe = "substantivo_masculino_singular"

                    palavras.append(Palavra(texto=texto_palavra, classe=classe))
                return palavras

        except Exception as e:
            erro = str(e).lower()
            if 'rate limit' in erro or '429' in erro:
                print(f"      ⚠️ {api['nome']} rate limit, trocando...")
            else:
                print(f"      ⚠️ {api['nome']} erro: {erro[:80]}")
            continue

    print(f"      ❌ Todas as APIs falharam!")
    return []

# Executar classificação
inicio_total = datetime.now()

for lang in ["pt", "en", "es", "fr"]:
    if lang not in pipeline.legendas_idiomas:
        continue

    legendas = pipeline.legendas_idiomas[lang]
    print(f"\n{'='*60}")
    print(f"🔤 {lang.upper()} - {len(legendas)} legendas")
    print(f"{'='*60}")

    for j, leg in enumerate(legendas):
        print(f"\n   [{j+1}/{len(legendas)}] {leg.texto[:50]}...")

        palavras = classificar_legenda(leg.texto, lang)

        if palavras:
            leg.palavras = palavras
            print(f"      ✅ {len(palavras)} palavras classificadas")
        else:
            print(f"      ❌ Falhou!")

        if j < len(legendas) - 1:
            print(f"      ⏳ Aguardando {DELAY_ENTRE_LEGENDAS}s...")
            time.sleep(DELAY_ENTRE_LEGENDAS)

    if lang != "fr":
        print(f"\n⏳ Aguardando {DELAY_ENTRE_IDIOMAS}s...")
        time.sleep(DELAY_ENTRE_IDIOMAS)

tempo_total = (datetime.now() - inicio_total).seconds
print("\n" + "=" * 70)
print(f"✅ CLASSIFICAÇÃO CONCLUÍDA!")
print(f"   ⏱️  Tempo total: {tempo_total // 60}min {tempo_total % 60}s")
print("=" * 70)

# Salvar JSONs
print("\n📦 SALVANDO CLASSIFICAÇÕES...")
for lang in ["pt", "en", "es", "fr"]:
    if lang not in pipeline.legendas_idiomas:
        continue

    dados = {}
    for leg in pipeline.legendas_idiomas[lang]:
        dados[str(leg.id)] = {
            'inicio': leg.inicio_str,
            'fim': leg.fim_str,
            'texto_original': leg.texto,
            'palavras': [{'texto': p.texto, 'classe': p.classe} for p in leg.palavras]
        }

    json_path = Path(f'classificacao_{config.NOME_ORACAO}_{lang}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)
    print(f"   ✅ classificacao_{config.NOME_ORACAO}_{lang}.json")

print("\n📦 Para gerar o pacote de revisão:")
print("   pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)")


🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA
   Chamando APIs diretamente (Mistral → Groq)
⏱️  Delay entre legendas: 2s
⏱️  Delay entre idiomas: 5s

🔤 PT - 7 legendas

   [1/7] Pai nosso que estais no céu, santificado seja o vo...
      🔄 Tentando: Mistral...
      ✅ Mistral respondeu!
      ✅ 11 palavras classificadas
      ⏳ Aguardando 2s...

   [2/7] Venha a nós o vosso reino, seja feita a vossa vont...
      🔄 Tentando: Mistral...
      ✅ Mistral respondeu!
      ✅ 11 palavras classificadas
      ⏳ Aguardando 2s...

   [3/7] Assim na terra como no céu. O pão nosso de cada di...
      🔄 Tentando: Mistral...
      ✅ Mistral respondeu!
      ✅ 12 palavras classificadas
      ⏳ Aguardando 2s...

   [4/7] Nos dai hoje. Perdoai as nossas ofensas....
      🔄 Tentando: Mistral...
      ✅ Mistral respondeu!
      ✅ 7 palavras classificadas
      ⏳ Aguardando 2s...

   [5/7] Assim como nós perdoamos a quem nos tem ofendido....
      🔄 Tentando: Mistral...
      ✅ Mistral respondeu!
      ✅ 9 palavr

📦 PACOTE DE REVISÃO
O que faz:

Copia JSONs para a pasta de correções (correcoes/{oracao}/)

Cria prompt_revisao.md e relatorio_classificacoes.csv (se não existirem)

⚠️ Executar APÓS a FASE 5A

In [24]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 EXECUTAR PACOTE DE REVISÃO (cria guias + copia JSONs)       ║
# ║                                                                 ║
# ║  ✅ Cria prompt_revisao.md e relatorio_classificacoes.csv       ║
# ║  ✅ Copia JSONs para a pasta correta no Drive                   ║
# ║  ✅ Salva também na pasta local                                ║
# ╚══════════════════════════════════════════════════════════════════╝

print("═" * 65)
print("📦 EXECUTANDO PACOTE DE REVISÃO")
print("═" * 65)

# Executar o método que cria os guias e copia os JSONs
pasta_json = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)

print("\n" + "═" * 65)
print("✅ PACOTE DE REVISÃO EXECUTADO COM SUCESSO!")
print("═" * 65)

print(f"\n📁 JSONs salvos em: {pasta_json}")
print(f"📁 Guias salvos em: /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/")

print("\n📄 ARQUIVOS CRIADOS:")
print("   📄 prompt_revisao.md")
print("   📄 relatorio_classificacoes.csv")

# Verificar se os arquivos foram criados (CAMINHO CORRIGIDO!)
from pathlib import Path
pasta_correcoes_root = Path('/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes')

print("\n🔍 VERIFICANDO ARQUIVOS:")
for arquivo in ['prompt_revisao.md', 'relatorio_classificacoes.csv']:
    caminho = pasta_correcoes_root / arquivo
    if caminho.exists():
        tamanho = caminho.stat().st_size / 1024
        print(f"   ✅ {arquivo} ({tamanho:.1f} KB)")
    else:
        print(f"   ❌ {arquivo} - erro na criação")

═════════════════════════════════════════════════════════════════
📦 EXECUTANDO PACOTE DE REVISÃO
═════════════════════════════════════════════════════════════════

═════════════════════════════════════════════════════════════════
✅ PACOTE DE REVISÃO EXECUTADO COM SUCESSO!
═════════════════════════════════════════════════════════════════

📁 JSONs salvos em: /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/pai_nosso
📁 Guias salvos em: /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/

📄 ARQUIVOS CRIADOS:
   📄 prompt_revisao.md
   📄 relatorio_classificacoes.csv

🔍 VERIFICANDO ARQUIVOS:
   ✅ prompt_revisao.md (5.3 KB)
   ✅ relatorio_classificacoes.csv (10.2 KB)


In [25]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 DOWNLOAD AUTOMÁTICO (com caminhos corrigidos)               ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil
import zipfile
from datetime import datetime
from google.colab import files

NOME_ORACAO = config.NOME_ORACAO

# CAMINHOS CORRETOS
pasta_correcoes_root = Path('/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes')
pasta_oracao = pasta_correcoes_root / NOME_ORACAO

print("═" * 65)
print("💾 PREPARANDO DOWNLOAD")
print("═" * 65)

# Criar pasta temporária
temp_dir = Path('/content/download_pacote')
temp_dir.mkdir(exist_ok=True)

# 1. Copiar guias
for guia in ['prompt_revisao.md', 'relatorio_classificacoes.csv']:
    src = pasta_correcoes_root / guia
    if src.exists():
        shutil.copy(src, temp_dir / guia)
        print(f"   ✅ {guia}")
    else:
        print(f"   ⚠️ {guia} - não encontrado")

# 2. Copiar JSONs (do local, que é o mais atual)
for lang in ['pt', 'en', 'es', 'fr']:
    src = Path(f'/content/classificacao_{NOME_ORACAO}_{lang}.json')
    if src.exists():
        shutil.copy(src, temp_dir / src.name)
        print(f"   ✅ {src.name}")
    else:
        print(f"   ❌ {src.name} - não encontrado")

# 3. Criar README
readme_content = f"""# PACOTE DE REVISÃO - {NOME_ORACAO.upper()}
## Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## INSTRUÇÕES:
1. Abra o prompt_revisao.md e copie o conteúdo
2. Cole numa IA (Claude, GPT, etc.)
3. Anexe os 4 JSONs
4. A IA retorna os JSONs corrigidos
5. Salve no: MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/{NOME_ORACAO}/
6. Execute a FASE 5C
"""
(temp_dir / "LEIA-ME.txt").write_text(readme_content, encoding='utf-8')
print(f"   ✅ LEIA-ME.txt")

# 4. Criar ZIP
zip_path = Path('/content') / f'pacote_revisao_{NOME_ORACAO}.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for arquivo in temp_dir.iterdir():
        zipf.write(arquivo, arquivo.name)

# 5. Limpar e baixar
shutil.rmtree(temp_dir)
files.download(str(zip_path))

print(f"\n✅ Download iniciado: pacote_revisao_{NOME_ORACAO}.zip")

═════════════════════════════════════════════════════════════════
💾 PREPARANDO DOWNLOAD
═════════════════════════════════════════════════════════════════
   ✅ prompt_revisao.md
   ✅ relatorio_classificacoes.csv
   ✅ classificacao_pai_nosso_pt.json
   ✅ classificacao_pai_nosso_en.json
   ✅ classificacao_pai_nosso_es.json
   ✅ classificacao_pai_nosso_fr.json
   ✅ LEIA-ME.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download iniciado: pacote_revisao_pai_nosso.zip


⏸️ FASE 5B — Revisão manual pela IA (você faz isso)
Após executar o PACOTE DE REVISÃO:

Abra o Google Drive → MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/

Copie o conteúdo do arquivo prompt_revisao.md

Cole numa IA (Claude, GPT, etc.) junto com os 4 JSONs que estão na pasta da sua oração (ex: pai_nosso/)

A IA retorna os JSONs corrigidos

Salve os JSONs corrigidos na mesma pasta (substituindo os originais)

Execute a FASE 5C abaixo

In [26]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5C — Recarregar do Drive após revisão                    ║
# ║  Execute APÓS salvar os JSONs corrigidos no Drive              ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📥 FASE 5C — RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS")
print("═" * 65)

# CAMINHO CORRIGIDO (repositório consolidado)
pasta_correcoes = Path(f'/content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/{config.NOME_ORACAO}')

if not pasta_correcoes.exists():
    print(f"❌ Pasta não encontrada: {pasta_correcoes}")
    print("   Verifique se os JSONs corrigidos foram salvos no lugar certo")
    print("   Caminho esperado: MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/pai_nosso/")
else:
    jsons_ok = []
    jsons_faltando = []
    for lang in config.IDIOMAS:
        p = pasta_correcoes / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
        if p.exists():
            jsons_ok.append(lang.upper())
            print(f"   ✅ {p.name}")
        else:
            jsons_faltando.append(lang.upper())
            print(f"   ❌ {p.name}")

    if len(jsons_ok) == 4:
        print("\n✅ Todos os 4 JSONs encontrados! Carregando...")
        pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

        print("\n📋 Preview das classificações:")
        print("─" * 55)
        for lang in config.IDIOMAS:
            legs = pipeline.legendas_idiomas.get(lang, [])
            if legs and legs[0].palavras:
                leg = legs[0]
                print(f'\n  {lang.upper()} — "{leg.texto[:50]}"')
                from constants import CORES_HTML
                for p in leg.palavras[:4]:
                    ok  = '✅' if p.classe in CORES_HTML else '❌'
                    print(f'    {ok} {p.texto:<18} {p.classe}')
                if len(leg.palavras) > 4:
                    print(f'       ... (+{len(leg.palavras)-4} palavras)')

        print("\n" + "═" * 65)
        print("🎬 Pronto — execute as Fases 6, 7 e 8")
        print("═" * 65)
    else:
        print(f"\n⚠️  Faltam: {', '.join(jsons_faltando)}")
        print(f"   Coloque os JSONs corrigidos em: {pasta_correcoes}")

═════════════════════════════════════════════════════════════════
📥 FASE 5C — RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS
═════════════════════════════════════════════════════════════════
   ❌ classificacao_pai_nosso_pt.json
   ✅ classificacao_pai_nosso_en.json
   ❌ classificacao_pai_nosso_es.json
   ❌ classificacao_pai_nosso_fr.json

⚠️  Faltam: PT, ES, FR
   Coloque os JSONs corrigidos em: /content/drive/MyDrive/pai_nosso_refatorado_v1/pipeline/correcoes/pai_nosso


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAR SE PALAVRAS FORAM ATRIBUÍDAS                      ║
# ╚══════════════════════════════════════════════════════════════════╝

print("═" * 80)
print("🔍 VERIFICANDO PALAVRAS APÓS ATRIBUIÇÃO")
print("═" * 80)

for lang in ["pt", "en", "es", "fr"]:
    if lang in pipeline.legendas_idiomas:
        leg0 = pipeline.legendas_idiomas[lang][0]
        print(f"\n📝 {lang.upper()}:")
        print(f"   Palavras classificadas: {len(leg0.palavras)}")

        if leg0.palavras:
            for p in leg0.palavras[:4]:
                print(f"      {p.texto:15} → {p.classe}")
        else:
            print(f"   ❌ Ainda sem palavras!")
    else:
        print(f"\n📝 {lang.upper()}: não encontrado")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAÇÃO COMPLETA - Texto integral e sincronia           ║
# ║  Mostra a oração inteira de cada idioma                         ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt

print("═" * 80)
print("🔍 VERIFICAÇÃO COMPLETA DOS 4 IDIOMAS")
print(f"   Oração: {config.NOME_ORACAO}")
print("═" * 80)

# Dicionário para armazenar textos completos
textos_completos = {}
timestamps_por_idioma = {}

for lang in ["pt", "en", "es", "fr"]:
    srt_path = Path(config.nome_srt(lang))
    if not srt_path.exists():
        print(f"\n❌ {lang.upper()}: arquivo não encontrado!")
        continue

    legendas = ler_srt(srt_path)

    print(f"\n{'='*80}")
    print(f"📝 {lang.upper()} - {len(legendas)} segmentos")
    print(f"{'='*80}")

    # Mostrar cada legenda com timestamp
    for leg in legendas:
        print(f"\n[{leg.id:02d}] {leg.inicio_str} → {leg.fim_str}")
        print(f"    {leg.texto}")

    # Juntar texto completo
    texto_completo = " ".join([leg.texto for leg in legendas])
    textos_completos[lang] = texto_completo

    # Guardar timestamps
    timestamps_por_idioma[lang] = [(leg.inicio_str, leg.fim_str) for leg in legendas]

# ═══════════════════════════════════════════════════════════════════
# MOSTRAR TEXTO COMPLETO DE CADA IDIOMA
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("📖 TEXTO COMPLETO POR IDIOMA")
print("═" * 80)

for lang in ["pt", "en", "es", "fr"]:
    if lang in textos_completos:
        print(f"\n{lang.upper()}:")
        print(f"   {textos_completos[lang]}")
        print(f"   Total caracteres: {len(textos_completos[lang])}")

# ═══════════════════════════════════════════════════════════════════
# VERIFICAR SINCRONIA DOS TIMESTAMPS
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("⏱️ VERIFICAÇÃO DE SINCRONIA (todos devem ter os mesmos timestamps)")
print("═" * 80)

if "pt" in timestamps_por_idioma:
    pt_timestamps = timestamps_por_idioma["pt"]
    print(f"\n✅ PT (REFERÊNCIA): {len(pt_timestamps)} segmentos")

    for lang in ["en", "es", "fr"]:
        if lang in timestamps_por_idioma:
            lang_timestamps = timestamps_por_idioma[lang]

            if len(pt_timestamps) != len(lang_timestamps):
                print(f"\n❌ {lang.upper()}: NÚMERO DIFERENTE! PT={len(pt_timestamps)} vs {lang.upper()}={len(lang_timestamps)}")
            else:
                diferentes = []
                for i, ((pt_ini, pt_fim), (lang_ini, lang_fim)) in enumerate(zip(pt_timestamps, lang_timestamps)):
                    if pt_ini != lang_ini or pt_fim != lang_fim:
                        diferentes.append((i+1, pt_ini, pt_fim, lang_ini, lang_fim))

                if diferentes:
                    print(f"\n⚠️ {lang.upper()}: DIFERENÇAS ENCONTRADAS!")
                    for i, pt_ini, pt_fim, lang_ini, lang_fim in diferentes:
                        print(f"      Legenda {i}: PT({pt_ini}→{pt_fim}) vs {lang.upper()}({lang_ini}→{lang_fim})")
                else:
                    print(f"\n✅ {lang.upper()}: TIMESTAMPS IGUAIS AO PT!")

# ═══════════════════════════════════════════════════════════════════
# VERIFICAR SE FALTA TEXTO (comparar com original)
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("📋 COMPARAÇÃO COM ORAÇÃO ORIGINAL")
print("═" * 80)

print(f"\n📝 ORAÇÃO ORIGINAL (config.TEXTO_ORACAO):")
print(f"   {config.TEXTO_ORACAO}")
print(f"   Total caracteres: {len(config.TEXTO_ORACAO)}")

if "pt" in textos_completos:
    print(f"\n📝 PT ATUAL:")
    print(f"   {textos_completos['pt']}")
    print(f"   Total caracteres: {len(textos_completos['pt'])}")

    if len(textos_completos['pt']) < len(config.TEXTO_ORACAO):
        print(f"\n⚠️ ATENÇÃO: PT está com {len(config.TEXTO_ORACAO) - len(textos_completos['pt'])} caracteres a MENOS!")
    elif len(textos_completos['pt']) > len(config.TEXTO_ORACAO):
        print(f"\n⚠️ ATENÇÃO: PT está com {len(textos_completos['pt']) - len(config.TEXTO_ORACAO)} caracteres a MAIS!")
    else:
        print(f"\n✅ PT tem o MESMO tamanho que o original!")

print("\n" + "═" * 80)
print("✅ VERIFICAÇÃO CONCLUÍDA!")
print("═" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 RESETAR E REGERAR VÍDEO DO ZERO                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt
from ffmpeg_utils import gerar_ass, queimar_legendas_ass

print("=" * 80)
print("🔧 RESETANDO E REGERANDO VÍDEO")
print("=" * 80)

# 1. Resetar checkpoint a partir de clipes_cortados
print("\n🔄 Resetando checkpoint...")
pipeline._cp.reiniciar_de('clipes_cortados')

# 2. Limpar pastas temporárias antigas
import shutil
for pasta in ['clipes_cortados', 'clipes_prontos', 'temp_raw']:
    p = Path(pasta)
    if p.exists():
        shutil.rmtree(p)
        print(f"   🗑️ Removido: {pasta}/")

# 3. Carregar legendas PT
legendas_pt = ler_srt(config.NOME_SRT_PT)
print(f"\n✅ PT: {len(legendas_pt)} segmentos")

# 4. FASE 6 - Baixar clipes
print("\n📹 FASE 6: Baixando clipes...")
clipes = pipeline.fase6_baixar_clipes(legendas_pt)
print(f"✅ {len(clipes)} clipes processados")

# 5. FASE 7 - Vídeo base
print("\n🎵 FASE 7: Criando vídeo base...")
video_base = pipeline.fase7_criar_video_base(clipes)
print(f"✅ {video_base}")

# 6. Verificar palavras classificadas
print("\n📝 VERIFICANDO PALAVRAS CLASSIFICADAS:")
for lang in ["pt", "en", "es", "fr"]:
    if lang in pipeline.legendas_idiomas:
        palavras = len(pipeline.legendas_idiomas[lang][0].palavras) if pipeline.legendas_idiomas[lang][0].palavras else 0
        print(f"   {lang.upper()}: {palavras} palavras na legenda 1")

# 7. FASE 8 - Gerar ASS e queimar legendas
print("\n🖊️ FASE 8: Gerando legendas coloridas...")
ass_path = gerar_ass(pipeline.legendas_idiomas, config)
print(f"✅ ASS gerado: {ass_path}")

# Verificar tags de cor
with open(ass_path, 'r', encoding='utf-8-sig') as f:
    conteudo = f.read()
    n_tags = conteudo.count('\\1c')
    print(f"   Tags de cor: {n_tags}")

# 8. Gerar vídeo final
video_final = Path(config.NOME_VIDEO_FINAL)
queimar_legendas_ass(video_base, ass_path, video_final)

print("\n" + "=" * 80)
print(f"🎉 VÍDEO FINAL: {video_final}")
if video_final.exists():
    print(f"   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB")
print("=" * 80)

# Visualizar
from IPython.display import Video, display
display(Video(str(video_final), embed=True, width=800))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 GERAR VÍDEO COM CORES POR PALAVRA                           ║
# ║  (Esta célula já inclui FASES 6, 7 e 8)                         ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from ffmpeg_utils import gerar_ass, queimar_legendas_ass
from srt_utils import ler_srt

print("═" * 80)
print("🎬 GERANDO VÍDEO COM CORES POR PALAVRA")
print("═" * 80)

# Carregar legendas PT
legendas_pt = ler_srt(config.NOME_SRT_PT)
print(f"✅ PT: {len(legendas_pt)} segmentos")

# FASE 6 - Clipes (se não existirem)
if not pipeline._cp.fase_concluida("clipes_cortados"):
    print("\n📹 Baixando clipes...")
    clipes = pipeline.fase6_baixar_clipes(legendas_pt)
    print(f"✅ {len(clipes)} clipes")
else:
    print("\n⏭️ Clipes já existem")
    clipes = pipeline._clipes_do_checkpoint()

# FASE 7 - Vídeo base (se não existir)
video_base = Path(config.NOME_VIDEO_BASE)
if not video_base.exists():
    print("\n🎵 Criando vídeo base...")
    video_base = pipeline.fase7_criar_video_base(clipes)
    print(f"✅ {video_base}")
else:
    print(f"\n⏭️ Vídeo base já existe: {video_base}")

# Verificar palavras antes de gerar ASS
print("\n📝 VERIFICANDO PALAVRAS:")
for lang in ["pt", "en", "es", "fr"]:
    if lang in pipeline.legendas_idiomas:
        n_palavras = len(pipeline.legendas_idiomas[lang][0].palavras)
        print(f"   {lang.upper()}: {n_palavras} palavras na legenda 1")

# FASE 8 - Gerar ASS e queimar legendas
print("\n🖊️ Gerando legendas coloridas...")
ass_path = gerar_ass(pipeline.legendas_idiomas, config)
print(f"✅ ASS gerado: {ass_path}")

# Verificar tags de cor
with open(ass_path, 'r', encoding='utf-8-sig') as f:
    conteudo = f.read()
    n_tags = conteudo.count('\\1c')
    print(f"   Tags de cor no ASS: {n_tags}")

if n_tags == 0:
    print("\n❌ ERRO: Nenhuma tag de cor encontrada!")
    print("   As legendas ficarão sem cores.")
else:
    print("\n✅ Tags de cor encontradas!")

# Queimar legendas
video_final = Path(config.NOME_VIDEO_FINAL)
queimar_legendas_ass(video_base, ass_path, video_final)

print("\n" + "═" * 80)
print(f"🎉 VÍDEO FINAL: {video_final}")
print(f"   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB")
print("═" * 80)

# Visualizar
from IPython.display import Video, display
display(Video(str(video_final), embed=True, width=800))

### ▶ Fases 6–8 — Vídeo final (rode após a Fase 5C)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASES 6, 7 e 8 — Vídeo final                 ║
# ╚══════════════════════════════════════════════════╝

video_final = pipeline.continuar()

if video_final:
    print(f'\n🎬 Vídeo final: {video_final}')
    print(f'   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB')
    print(f'   Salvo no Drive: pasta Vídeos')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 DIAGNÓSTICO - Verificar estado do pipeline                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 80)
print("🔍 DIAGNÓSTICO - PIPELINE")
print("═" * 80)

# 1. Verificar checkpoint
print("\n📌 CHECKPOINT:")
print(pipeline._cp.resumo())

# 2. Verificar quais fases estão concluídas
print("\n📊 FASES CONCLUÍDAS:")
for fase in ["audio_gerado", "srt_pt_bruto", "srt_pt_corrigido",
             "srt_traduzidos", "classificacoes_feitas", "clipes_cortados",
             "video_base_criado", "legendas_queimadas"]:
    status = "✅" if pipeline._cp.fase_concluida(fase) else "❌"
    print(f"   {status} {fase}")

# 3. Verificar arquivos necessários
print("\n📁 ARQUIVOS NECESSÁRIOS:")

# Verificar SRTs
for lang in ["pt", "en", "es", "fr"]:
    srt = Path(f'{config.NOME_ORACAO}_{lang}.srt')
    status = "✅" if srt.exists() else "❌"
    print(f"   {status} {srt.name}")

# Verificar vídeo base
video_base = Path(config.NOME_VIDEO_BASE)
status = "✅" if video_base.exists() else "❌"
print(f"   {status} {config.NOME_VIDEO_BASE}")

# Verificar vídeo final
video_final = Path(config.NOME_VIDEO_FINAL)
status = "✅" if video_final.exists() else "❌"
print(f"   {status} {config.NOME_VIDEO_FINAL}")

# 4. Verificar legendas no pipeline
print("\n📝 LEGENDAS NO PIPELINE:")
if hasattr(pipeline, 'legendas_idiomas') and pipeline.legendas_idiomas:
    for lang, legendas in pipeline.legendas_idiomas.items():
        print(f"   {lang.upper()}: {len(legendas)} segmentos")
else:
    print("   ❌ pipeline.legendas_idiomas vazio!")

print("\n" + "═" * 80)

### 🚀 RUN ALL — Pipeline completo com checkpoint

Use quando quiser rodar tudo de uma vez. O pipeline pausa automaticamente na Fase 5A
se houver classificações a revisar. Após a revisão, rode a célula da Fase 5C
e depois a célula das Fases 6–8.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  RUN ALL                                       ║
# ╚══════════════════════════════════════════════════╝
from video_pipeline import ClassificacaoPendenteError

resultado = pipeline.run(
    # from_phase='clipes_cortados'  # descomente para retomar de uma fase
)

if resultado is None:
    print('\n⏸️  Pipeline pausado na Fase 5A.')
    print('   Siga as instruções acima, depois execute a célula FASE 5C.')
else:
    print(f'\n🎉 Vídeo final: {resultado}')
    print(pipeline._cp.resumo())

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 VISUALIZAR VÍDEO FINAL NO COLAB                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from IPython.display import Video, display

# Caminho do vídeo final
video_path = Path(config.NOME_VIDEO_FINAL)

if video_path.exists():
    print("═" * 65)
    print("🎬 VÍDEO FINAL")
    print("═" * 65)
    print(f"\n📁 Arquivo: {video_path.name}")
    print(f"📏 Tamanho: {video_path.stat().st_size / 1_048_576:.1f} MB")
    print("\n🎬 Reproduzindo vídeo:\n")

    # Mostrar o vídeo
    display(Video(str(video_path), embed=True, width=800))

    print("\n" + "═" * 65)
    print("📂 Links:")
    print(f"   Local: /content/{video_path.name}")
    print(f"   Drive: {config.pasta_assets_videos}")
    print("═" * 65)
else:
    print(f"❌ Vídeo não encontrado: {video_path}")
    print("   Execute as FASES 6, 7 e 8 primeiro")

### 🔧 Utilitários

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  UTILITÁRIOS — descomente o que precisar       ║
# ╚══════════════════════════════════════════════════╝
from checkpoint import Checkpoint
cp = Checkpoint()
print(cp.resumo())
print()
pipeline._clf.imprimir_status()

# ── Checkpoint ────────────────────────────────────────────────────────────────
# cp.resetar_tudo()
# cp.reiniciar_de('classificacoes_feitas')

# ── Classificações ────────────────────────────────────────────────────────────
# pipeline._clf.imprimir_status()
# pipeline._clf.classificar_idioma(pipeline.legendas_idiomas['en'], 'en', forcar=True)

# Gerar pacote de revisão manualmente (sem reclassificar):
# pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)
# print(f"📦 Pacote: {pacote}")

# Verificar classes inválidas:
# from constants import CORES_HTML
# for lang, legendas in pipeline.legendas_idiomas.items():
#     for leg in legendas:
#         for p in leg.palavras:
#             if p.classe not in CORES_HTML:
#                 print(f'  [{lang.upper()} leg.{leg.id}] "{p.texto}" → {p.classe} ❌')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 CORREÇÃO RÁPIDA - FASE 3 MANUAL                            ║
# ║  Execute esta célula para corrigir a segmentação               ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from srt_utils import ler_srt, salvar_srt
import re

print("═" * 70)
print("🔧 CORRIGINDO FASE 3 MANUALMENTE")
print("═" * 70)

# 1. Carregar timestamps do Whisper (mestre)
srt_whisper = Path(f'{config.NOME_ORACAO}_pt_whisper.srt')
if not srt_whisper.exists():
    print("❌ Whisper não encontrado. Execute a FASE 2 primeiro!")
else:
    legendas_timestamps = ler_srt(srt_whisper)
    print(f"✅ Timestamps do Whisper: {len(legendas_timestamps)} segmentos")

# 2. Carregar texto do YouTube
srt_youtube = Path(config.nome_srt('pt'))
if not srt_youtube.exists():
    print("❌ YouTube não encontrado. Execute a célula B0 primeiro!")
else:
    legendas_youtube = ler_srt(srt_youtube)
    print(f"✅ Texto do YouTube: {len(legendas_youtube)} segmentos")

    # Juntar todo o texto do YouTube
    texto_completo = " ".join([leg.texto for leg in legendas_youtube])
    print(f"✅ Texto completo: {len(texto_completo)} caracteres")

# 3. Redistribuir o texto nos timestamps do Whisper
n_whisper = len(legendas_timestamps)
total_chars = len(texto_completo)
chars_por_legenda = total_chars / n_whisper

print(f"\n📊 Redistribuindo {total_chars} caracteres em {n_whisper} segmentos...")

novas_legendas = []
pos = 0

for i, leg in enumerate(legendas_timestamps):
    # Calcula onde termina esta fatia
    fim_pos = int(pos + chars_por_legenda)

    # Ajusta para não cortar no meio de uma palavra
    if fim_pos < total_chars:
        while fim_pos < total_chars and texto_completo[fim_pos] != ' ':
            fim_pos += 1

    # Pega o texto da fatia
    texto_parte = texto_completo[pos:fim_pos].strip()

    # Última legenda pega o resto todo
    if i == n_whisper - 1:
        texto_parte = texto_completo[pos:].strip()

    # Atualiza o texto da legenda
    leg.texto = texto_parte
    novas_legendas.append(leg)

    pos = fim_pos

# 4. Salvar o SRT corrigido
srt_pt = Path(config.NOME_SRT_PT)
salvar_srt(novas_legendas, srt_pt)

print("\n" + "═" * 70)
print(f"✅ FASE 3 CORRIGIDA! | {len(novas_legendas)} legendas")
print("═" * 70)
print("\n📋 RESULTADO:")
for leg in novas_legendas:
    print(f'  [{leg.id:02d}]  {leg.inicio_str} → {leg.fim_str}  |  {leg.texto[:70]}')
print("═" * 70)

print("\n✅ Agora execute as FASES 4, 5A, etc. normalmente!")

PONTOS A CORRIGIR NO PÓS MIGRAÇÃO:

### 📋 Status (revisão v1):

| Problema | Local | Status |
|----------|-------|-----------|
| **1. JSONs salvos no lugar errado** | `classification.py` (método `exportar_pacote_revisao`) | ✅ Corrigido — já salva em `pai_nosso_refatorado_v1/pipeline/correcoes/` |
| **2. Guias salvos no lugar certo** | `classification.py` | ✅ OK |
| **3. Verificação da célula** | Notebook (PACOTE REVISÃO) | ✅ Caminho correto |
| **4. Segmentação PT** | FASE 3 | PT tem 12 legendas (deveria ser 11) — pendente |
